# 06 — Denoising front-end experiment (bounded; candidate weights and threshold untouched)

Run 20261024T151905Z: the candidate's EER goes from 2.46 % (clean) to 15.98 % under white noise at 10 dB SNR → ratio 6.50 > 2.5 (**FAIL**).
This notebook tests whether a **lightweight denoiser in front of the detector** fixes that *without* destroying the synthesis artefacts the
detector relies on. It is an experiment, not an assumed fix. Noise stays a requirement; the threshold stays **0.5**; the checkpoint is unchanged.

**Stage A — development (already-inspected 244 clips; development evidence only)**
- Front-ends: `none`, `nr_stationary` (spectral gating, pure DSP), `deepfilternet3` (2 M-param neural denoiser, if it installs). Fixed settings, see `frontend.py`.
- The same front-end is applied to **every** input: genuine and synthetic, **clean and degraded** (clean → front-end → detector is exactly the deployed path).
- Paired per clip across required conditions {clean, 16→8→16 kHz, Opus 16 kb/s, MP3 32 kb/s, white noise 10 dB}. Per-condition rejections/coverage reported.
- **Artefact-removal check:** clean EER with front-end vs without, and how clean *synthetic* scores move (a front-end that lowers spoof scores on clean audio is eating artefacts).
- **Pre-registered freeze rule** (`eval_lib.frontend_selection`): eligible only if dev clean EER ≤ baseline clean EER + `DEV_SELECTION_MARGIN_PP`; among eligible, lowest worst-degraded EER; ties → simplest; frozen only if it beats the baseline's worst EER. Otherwise **no front-end is frozen** and the experiment is declared failed → fine-tuning (notebook 07, not run here).

**Stage B — fresh evaluation (new clips, no decisions)**
- Selection seed `20261028`, 150 + 150, speaker round-robin, **excluding all 500 inspected clip IDs** (200 pilot + 300 held-out). In-the-Wild shares all 54 speakers
  with the inspected sets (speaker-disjoint impossible) and the candidate's training data includes VoxCeleb2 celebrity speech: both limitations apply to every number.
- Baseline (`none`) and the frozen front-end on the five required conditions. Reported: absolute EERs with CIs, coverage, **unrounded** worst/clean ratio,
  PASS/FAIL/INCOMPLETE, **and** a clean-margin check (fresh clean EER with front-end must be within `OFFICIAL_MARGIN_PP` of the fresh baseline clean EER; a higher
  clean EER is never allowed to make the ratio look better).

**Margins.** The requirement we were given says Phase-1 EERs must "remain within 2 points"; we interpret that **provisionally as 2 percentage points**
(`OFFICIAL_MARGIN_PP = 2.0`, pending organiser clarification) and apply it to the Stage B clean-margin check. Stage A additionally uses a **stricter
development selection margin** of 1 pp (`DEV_SELECTION_MARGIN_PP = 1.0`) to choose the front-end; that is our own choice, **not** the official requirement.

Same T4 runtime as notebooks 04/05 preferred (model + cached audio reused); standalone also works. ≈ 25–35 min.

In [ ]:
# 1. Settings (fixed before any score is seen)
FRONTENDS = ["none", "nr_stationary", "deepfilternet3"]   # 'none' is the baseline; order = simplicity order for tie-breaks
TRY_DEEPFILTERNET = True        # pip install deepfilternet; marked unavailable if it fails (never silently skipped)
OFFICIAL_MARGIN_PP = 2.0        # PROVISIONAL interpretation of "Phase-1 EERs must remain within 2 points" = 2 percentage points (pending organiser clarification)
DEV_SELECTION_MARGIN_PP = 1.0   # stricter development-only margin for choosing a front-end at Stage A; NOT the official requirement
FROZEN_THRESHOLD = 0.5          # retained: no validation-selected threshold exists
REQUIRED = ["clean", "narrowband_8k", "codec_1", "codec_2", "noise"]
FRESH_SEED = 20261028
FRESH_N = 150                   # per class
N_BOOT = 500
WORK = "/content/vcd_eval"
USE_DRIVE = False
DRIVE_DIR = "/content/drive/MyDrive/vcd_cache"

In [ ]:
# 2. Dependencies
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "-q", "install", "onnxruntime", "librosa", "soundfile", "huggingface_hub",
                "transformers", "safetensors", "scipy", "noisereduce"], check=True)
DFN_INSTALL = {"attempted": TRY_DEEPFILTERNET}
if TRY_DEEPFILTERNET:
    r = subprocess.run([sys.executable, "-m", "pip", "-q", "install", "deepfilternet"], capture_output=True, text=True)
    DFN_INSTALL.update(returncode=r.returncode, tail=(r.stderr or r.stdout)[-400:])
print(DFN_INSTALL)

In [ ]:
%%writefile dhwani_infer.py
"""Inference wrapper for the third-party Dhwani ONNX checkpoint.

Model: ayush2635/Dhwani-Multilingual-Deepfake-Audio-Detection-Model (MIT),
file best_model.onnx at HF revision MODEL_REVISION. We did NOT train it.

Preprocessing is reproduced from the upstream training notebook
(github.com/ayush2635/Dhwani-Deepfake-Audio-Detection-API @ efd512c,
training/training_notebook.ipynb cell 4), which is the notebook whose cell 14
exports best_model.onnx with a (1, 64000) dummy input. See
docs/UPSTREAM_AUDIT.md for why we do not use the model card's z-score recipe
as the default.

Scores are softmax outputs of an UNCALIBRATED classifier. They are model
scores, not probabilities that a clip is synthetic.
"""

from __future__ import annotations

import hashlib
import json
import time
from dataclasses import dataclass, field, asdict

import numpy as np

# ---- pinned provenance -----------------------------------------------------
MODEL_REPO = "ayush2635/Dhwani-Multilingual-Deepfake-Audio-Detection-Model"
MODEL_FILE = "best_model.onnx"
MODEL_REVISION = "ee0187142973e1f02050e47d2a31c55ee0263065"
MODEL_SHA256 = "d1c232bf4d7990526804b375c15804c0ee4e9b566478b908c5a57431491b7342"
MODEL_BYTES = 1262760439

# ---- preprocessing / windowing policy ---------------------------------------
SR = 16000
WIN = 64000            # 4.0 s: training crop length and ONNX export shape
HOP = 32000            # 2.0 s: same hop as the upstream API's sliding window
MIN_DUR_S = 2.0        # brief: 2-10 s clips
MAX_DUR_S = 10.0       # longer clips: analyse the first 10 s, flagged
PEAK_TARGET_DBFS = -3.0

# ---- rejection gates (input validation only; never fed to the model) --------
SILENT_PEAK_DBFS = -50.0   # whole clip quieter than this -> silent
ACTIVE_FRAME_S = 0.025
ACTIVE_REL_DB = 35.0       # frame counts as active if within 35 dB of loudest frame
ACTIVE_ABS_DBFS = -55.0    # ...and above this absolute level
MIN_ACTIVE_S = 1.0

# Class order from upstream dataset.py: labels = [0]*real + [1]*fake.
# Documented in source; must still be confirmed empirically on known clips.
LABELS = {0: "genuine", 1: "synthetic"}
SYNTHETIC_INDEX = 1
DECISION_THRESHOLD = 0.5   # placeholder, NOT tuned on any validation set

# Controlled comparison (docs/UPSTREAM_AUDIT.md):
#   train_peak   - training recipe: peak -3 dBFS, 4 s windows             (A)
#   train_zscore - identical windows, z-score instead of peak norm        (A vs B isolates normalisation)
#   api_replica  - deployed API: first 3 s, z-score, zero-pad to 4 s     (B vs C isolates crop/padding)
PREPROCESSING_MODES = ("train_peak", "train_zscore", "api_replica")


class AudioRejected(Exception):
    """Input cannot be analysed (unreadable, empty, silent, too short)."""


def sha256_file(path: str, chunk: int = 1 << 22) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            b = f.read(chunk)
            if not b:
                break
            h.update(b)
    return h.hexdigest()


def _dbfs(x: float) -> float:
    return float(20 * np.log10(max(x, 1e-12)))


# ---- loading ------------------------------------------------------------------
def load_audio(path: str):
    """Return (samples [channels, n] float32 at native rate, native_sr).

    Tries soundfile first (wav/flac/ogg), then librosa/audioread (mp3, m4a, opus
    via ffmpeg). Never substitutes zeros on failure.
    """
    import soundfile as sf

    try:
        y, sr = sf.read(path, dtype="float32", always_2d=True)  # (n, ch)
        y = y.T
    except Exception as sf_err:
        try:
            import warnings

            import librosa

            with warnings.catch_warnings():  # expected audioread fallback for m4a/aac
                warnings.simplefilter("ignore", (UserWarning, FutureWarning))
                y, sr = librosa.load(path, sr=None, mono=False)
            y = np.atleast_2d(y).astype(np.float32)
        except Exception as lr_err:
            raise AudioRejected(
                f"unreadable audio (soundfile: {type(sf_err).__name__}; "
                f"librosa: {type(lr_err).__name__}: {lr_err})"
            ) from lr_err
    if y.size == 0 or y.shape[-1] == 0:
        raise AudioRejected("empty audio (0 samples)")
    if not np.all(np.isfinite(y)):
        raise AudioRejected("audio contains NaN/Inf samples")
    return y, int(sr)


def to_mono_16k(y: np.ndarray, sr: int) -> np.ndarray:
    mono = y.mean(axis=0) if y.shape[0] > 1 else y[0]
    if sr != SR:
        import librosa

        mono = librosa.resample(mono, orig_sr=sr, target_sr=SR, res_type="soxr_hq")
    return mono.astype(np.float32)


def signal_stats(x: np.ndarray) -> dict:
    peak = float(np.max(np.abs(x))) if x.size else 0.0
    rms = float(np.sqrt(np.mean(x.astype(np.float64) ** 2))) if x.size else 0.0
    flen = int(ACTIVE_FRAME_S * SR)
    nfr = len(x) // flen
    if nfr:
        fr = x[: nfr * flen].reshape(nfr, flen).astype(np.float64)
        fdb = 10 * np.log10(np.mean(fr**2, axis=1) + 1e-20)
        active = (fdb > fdb.max() - ACTIVE_REL_DB) & (fdb > ACTIVE_ABS_DBFS)
        active_s = float(active.sum() * ACTIVE_FRAME_S)
    else:
        active_s = 0.0
    return {
        "peak_dbfs": round(_dbfs(peak), 2),
        "rms_dbfs": round(_dbfs(rms), 2),
        "active_s": round(active_s, 3),
        "clipping_frac": round(float(np.mean(np.abs(x) >= 0.999)) if x.size else 0.0, 5),
    }


def validate(x16: np.ndarray) -> tuple[np.ndarray, dict]:
    """Apply the 2-10 s policy and silence gates. Returns (analysed audio, info)."""
    dur = len(x16) / SR
    info = {"duration_s": round(dur, 3), "truncated": False}
    if dur < MIN_DUR_S:
        raise AudioRejected(f"too short: {dur:.2f} s < {MIN_DUR_S} s minimum")
    if dur > MAX_DUR_S:
        x16 = x16[: int(MAX_DUR_S * SR)]
        info["truncated"] = True
    info["analysed_duration_s"] = round(len(x16) / SR, 3)
    st = signal_stats(x16)
    info.update(st)
    if st["peak_dbfs"] < SILENT_PEAK_DBFS:
        raise AudioRejected(f"silent: peak {st['peak_dbfs']} dBFS < {SILENT_PEAK_DBFS} dBFS")
    if st["active_s"] < MIN_ACTIVE_S:
        raise AudioRejected(
            f"mostly silent: {st['active_s']:.2f} s active signal < {MIN_ACTIVE_S} s"
        )
    return x16, info


# ---- preprocessing --------------------------------------------------------------
def window_starts(n: int) -> list[int]:
    """4 s windows, 2 s hop, last window aligned to the clip end (full coverage)."""
    if n <= WIN:
        return [0]
    starts = list(range(0, n - WIN + 1, HOP))
    if starts[-1] + WIN < n:
        starts.append(n - WIN)
    return starts


def make_windows(x16: np.ndarray, mode: str) -> tuple[np.ndarray, list[int]]:
    """Return (windows [k, WIN] float32, start sample indices)."""
    if mode in ("train_peak", "train_zscore"):
        if mode == "train_peak":
            # Training notebook: peak-normalise the clip to -3 dBFS, then crop/zero-pad
            # the END to 64000 samples. No mean/variance normalisation.
            peak = float(np.max(np.abs(x16)))
            x = x16 * ((10 ** (PEAK_TARGET_DBFS / 20)) / (peak + 1e-6))
        else:
            # Same windows; z-score over the analysed clip (before padding).
            x = (x16 - x16.mean()) / np.sqrt(x16.var() + 1e-5)
        starts = window_starts(len(x))
        wins = []
        for s in starts:
            w = x[s : s + WIN]
            if len(w) < WIN:
                w = np.pad(w, (0, WIN - len(w)))
            wins.append(w)
        return np.stack(wins).astype(np.float32), starts
    if mode == "api_replica":
        # Deployed upstream API (src/api/inference.py): first 48000 samples,
        # zero-pad to 48000, z-score, then zero-pad to 64000 in predict().
        # Kept only as a comparison; it differs from training.
        w = x16[:48000]
        w = np.pad(w, (0, 48000 - len(w)))
        w = (w - w.mean()) / np.sqrt(w.var() + 1e-5)
        w = np.pad(w, (0, WIN - len(w)))
        return w[None, :].astype(np.float32), [0]
    raise ValueError(f"unknown preprocessing mode {mode!r}; use {PREPROCESSING_MODES}")


def softmax(z: np.ndarray) -> np.ndarray:
    z = z - z.max(axis=-1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=-1, keepdims=True)


# ---- model ------------------------------------------------------------------------
def load_session(model_path: str, verify_sha: bool = True):
    import onnxruntime as ort

    if verify_sha:
        got = sha256_file(model_path)
        if got != MODEL_SHA256:
            raise RuntimeError(f"model sha256 mismatch: {got} != pinned {MODEL_SHA256}")
    so = ort.SessionOptions()
    sess = ort.InferenceSession(model_path, sess_options=so, providers=["CPUExecutionProvider"])
    return sess


def describe_session(sess) -> dict:
    import onnxruntime as ort

    return {
        "onnxruntime_version": ort.__version__,
        "available_providers": ort.get_available_providers(),
        "session_providers": sess.get_providers(),
        "inputs": [(i.name, i.shape, i.type) for i in sess.get_inputs()],
        "outputs": [(o.name, o.shape, o.type) for o in sess.get_outputs()],
    }


def interface_self_test(sess, seed: int = 0) -> dict:
    """Probe what the checkpoint actually accepts (metadata alone proves nothing)."""
    rng = np.random.default_rng(seed)
    name = sess.get_inputs()[0].name
    out = {}
    base = (rng.standard_normal(WIN) * 0.1).astype(np.float32)
    ref = sess.run(None, {name: base[None]})[0]
    out["len_64000_batch1"] = {"ok": True, "shape": list(ref.shape)}
    for n in (32000, 48000, 96000):
        try:
            r = sess.run(None, {name: base[:n][None] if n <= WIN else np.tile(base, 2)[:n][None]})[0]
            out[f"len_{n}_batch1"] = {"ok": True, "shape": list(r.shape)}
        except Exception as e:
            out[f"len_{n}_batch1"] = {"ok": False, "error": f"{type(e).__name__}: {str(e)[:200]}"}
    try:
        b = np.stack([base, base[::-1].copy()])
        r = sess.run(None, {name: b})[0]
        r1 = sess.run(None, {name: base[::-1].copy()[None]})[0]
        # Colab run 20261024T102407Z: batch of 2 returned shape [1, 2] (a row is
        # silently dropped), so "ok" requires one output row per input row.
        out["batch2"] = {
            "ok": r.shape[0] == 2,
            "shape": list(r.shape),
            "max_abs_diff_vs_individual": float(
                max(np.abs(r[0] - ref[0]).max(), np.abs(r[1] - r1[0]).max())
            ) if r.shape[0] == 2 else None,
        }
    except Exception as e:
        out["batch2"] = {"ok": False, "error": f"{type(e).__name__}: {str(e)[:200]}"}
    # determinism
    again = sess.run(None, {name: base[None]})[0]
    out["deterministic"] = bool(np.allclose(again, ref))
    return out


@dataclass
class Result:
    file: str
    status: str = "ok"
    reject_reason: str = ""
    file_sha256: str = ""
    native_sr: int | None = None
    channels: int | None = None
    info: dict = field(default_factory=dict)
    preprocessing: str = ""
    window_starts_s: list = field(default_factory=list)
    window_logits: list = field(default_factory=list)
    window_scores: list = field(default_factory=list)
    clip_score_mean: float | None = None
    clip_score_max: float | None = None
    predicted_label: str = ""
    latency_ms_model: float | None = None
    latency_ms_total: float | None = None


def analyse(sess, path: str, mode: str = "train_peak") -> Result:
    """Score an audio file (any format load_audio can decode)."""
    t0 = time.perf_counter()
    res = Result(file=path, preprocessing=mode)
    try:
        res.file_sha256 = sha256_file(path)
        y, sr = load_audio(path)
    except AudioRejected as e:
        res.status, res.reject_reason = "rejected", str(e)
        res.latency_ms_total = round((time.perf_counter() - t0) * 1000, 1)
        return res
    return _score(sess, res, y, sr, t0)


def analyse_waveform(sess, samples: np.ndarray, sr: int, mode: str = "train_peak",
                     label: str = "<waveform>") -> Result:
    """Score in-memory PCM ([n] or [channels, n], float in [-1, 1]).

    Same validation, preprocessing and windowing as analyse(); intended for
    audio windows that never touch disk (e.g. remote-call PCM later).
    """
    t0 = time.perf_counter()
    res = Result(file=label, preprocessing=mode)
    y = np.atleast_2d(np.asarray(samples, dtype=np.float32))
    if y.size == 0:
        res.status, res.reject_reason = "rejected", "empty audio (0 samples)"
    elif not np.all(np.isfinite(y)):
        res.status, res.reject_reason = "rejected", "audio contains NaN/Inf samples"
    if res.status == "rejected":
        res.latency_ms_total = round((time.perf_counter() - t0) * 1000, 1)
        return res
    return _score(sess, res, y, int(sr), t0)


def _score(sess, res: Result, y: np.ndarray, sr: int, t0: float) -> Result:
    try:
        res.native_sr, res.channels = sr, int(y.shape[0])
        x16 = to_mono_16k(y, sr)
        x16, info = validate(x16)
        res.info = info
        wins, starts = make_windows(x16, res.preprocessing)
    except AudioRejected as e:
        res.status, res.reject_reason = "rejected", str(e)
        res.latency_ms_total = round((time.perf_counter() - t0) * 1000, 1)
        return res

    name = sess.get_inputs()[0].name
    logits = []
    tm = time.perf_counter()
    for w in wins:  # batch=1 only: batch>1 drops rows (see interface_self_test)
        logits.append(sess.run(None, {name: w[None]})[0][0])
    res.latency_ms_model = round((time.perf_counter() - tm) * 1000, 1)
    logits = np.asarray(logits, dtype=np.float64)
    scores = softmax(logits)[:, SYNTHETIC_INDEX]
    res.window_starts_s = [round(s / SR, 3) for s in starts]
    res.window_logits = [[round(float(a), 4) for a in row] for row in logits]
    res.window_scores = [round(float(s), 4) for s in scores]
    res.clip_score_mean = round(float(scores.mean()), 4)
    res.clip_score_max = round(float(scores.max()), 4)
    res.predicted_label = LABELS[SYNTHETIC_INDEX] if res.clip_score_mean >= DECISION_THRESHOLD else LABELS[0]
    res.latency_ms_total = round((time.perf_counter() - t0) * 1000, 1)
    return res


def result_row(res: Result, **extra) -> dict:
    d = asdict(res)
    info = d.pop("info")
    for k in ("window_starts_s", "window_logits", "window_scores"):
        d[k] = json.dumps(d[k])
    row = {**extra, **d, **info}
    row.update(
        decision_threshold=DECISION_THRESHOLD,
        label_mapping=json.dumps(LABELS),
        model_repo=MODEL_REPO,
        model_revision=MODEL_REVISION,
        model_sha256=MODEL_SHA256,
    )
    return row

In [ ]:
%%writefile eval_lib.py
"""Helpers for the Dhwani generalisation evaluation (evaluation only, no training).

Standard library only, so the selection, archive access and metric code can be
unit-tested without ML dependencies (evaluation/tests/test_eval_lib.py).
Model execution, preprocessing and windowing are NOT reimplemented here: the
notebook calls src/vcd/dhwani_infer.py (embedded verbatim, read-only).

Labels follow dhwani_infer.LABELS: "genuine" (0) / "synthetic" (1).
Scores are uncalibrated synthetic-class softmax means, not probabilities.
"""

from __future__ import annotations

import csv
import hashlib
import io
import json
import math
import random
import time
import urllib.error
import urllib.request
from collections import Counter, defaultdict

# ---- pinned dataset sources (verified 2026-10-24, see evaluation/README.md) --------
SOURCES = {
    "asvspoof2019_la_eval": {
        "title": "ASVspoof 2019 Logical Access (LA), evaluation partition",
        "landing_page": "https://datashare.ed.ac.uk/handle/10283/3336",
        "doi": "10.7488/ds/2555",
        "archive": "LA.zip",
        "url": "https://datashare.ed.ac.uk/server/api/core/bitstreams/"
               "a9f87c35-f055-4015-80e2-2fdff0d46269/content",
        "bytes": 7640952520,
        "md5": "30c98f11d8b2bc21f2c257bfd78bb5c5",  # DSpace checkSum (MD5) == HTTP ETag
        "last_modified": "Tue, 04 Jun 2019 09:38:20 GMT",
        "license": "ODC-By 1.0 (archive README.txt, COPYING section)",
        "protocol_suffix": "ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.eval.trl.txt",
        "audio_dir_suffix": "ASVspoof2019_LA_eval/flac/",
        "audio_ext": ".flac",
        # Literature values, checked against the protocol in the notebook (warn on mismatch).
        "expected_counts": {"genuine": 7355, "synthetic": 63882},
        "expected_attacks": [f"A{i:02d}" for i in range(7, 20)],
    },
    "in_the_wild": {
        "title": "In-the-Wild audio deepfake dataset (Mueller et al., Interspeech 2022)",
        "landing_page": "https://huggingface.co/datasets/mueller91/In-The-Wild",
        "project_page": "https://deepfake-total.com/in_the_wild",
        "hf_repo": "mueller91/In-The-Wild",
        "revision": "eee168f92c367f8c82ff2cf42b6f61e362fd6211",
        "archive": "release_in_the_wild.zip",
        "url": "https://huggingface.co/datasets/mueller91/In-The-Wild/resolve/"
               "eee168f92c367f8c82ff2cf42b6f61e362fd6211/release_in_the_wild.zip",
        "bytes": 8161489023,
        "sha256": "46665f30a6758b45642c659c7e0e80da2ba3d121b064716824a56a010ddf9e1a",  # LFS oid
        "xet_hash": "ec530d2076f973d40a8e388e559888f1b4fea467db8126c811d34dd508ee106f",  # CDN ETag
        "license": "CONFLICT: HF card says CC-BY-SA-4.0; Fraunhofer project page says Apache-2.0",
        "meta_suffix": "meta.csv",
        "expected_counts": {"genuine": 19963, "synthetic": 11816},
    },
}

GENUINE, SYNTHETIC = "genuine", "synthetic"


# ---- remote zip access via HTTP range requests -------------------------------------
class RangeUnsupported(RuntimeError):
    """Server ignored the Range header (would have streamed the whole archive)."""


class HTTPRangeFile(io.RawIOBase):
    """Read-only, seekable view of a remote file using HTTP Range requests.

    zipfile.ZipFile can open it directly, so only the central directory and the
    selected members are transferred. zipfile checks each member's CRC-32 on read.
    `fetch(start, end_inclusive) -> bytes` can be injected for tests.
    """

    def __init__(self, url: str | None = None, size: int | None = None, fetch=None,
                 timeout: float = 60.0, retries: int = 4):
        super().__init__()
        self.url, self.timeout, self.retries = url, timeout, retries
        self._final_url = None
        self._fetch = fetch or self._http_fetch
        self.headers = {}
        if size is None:
            size = self._probe_size()
        self.size, self.pos = int(size), 0
        self.n_requests, self.bytes_fetched = 0, 0

    def _resolve(self):
        req = urllib.request.Request(self.url, headers={"Range": "bytes=0-0"})
        with urllib.request.urlopen(req, timeout=self.timeout) as r:
            if r.status != 206:
                raise RangeUnsupported(f"HTTP {r.status} to a Range request: {self.url}")
            self._final_url = r.geturl()
            self.headers = {k.lower(): v for k, v in r.headers.items()}
            r.read()
        return self.headers

    def _probe_size(self) -> int:
        h = self._resolve()
        return int(h["content-range"].rsplit("/", 1)[1])

    def _http_fetch(self, start: int, end: int) -> bytes:
        last = None
        for attempt in range(self.retries):
            if self._final_url is None:
                self._resolve()
            req = urllib.request.Request(self._final_url, headers={"Range": f"bytes={start}-{end}"})
            try:
                with urllib.request.urlopen(req, timeout=self.timeout) as r:
                    if r.status != 206:
                        raise RangeUnsupported(f"HTTP {r.status} to a Range request")
                    data = r.read()
                if len(data) != end - start + 1:
                    raise IOError(f"short read {len(data)} != {end - start + 1}")
                return data
            except RangeUnsupported:
                raise
            except (urllib.error.URLError, IOError, TimeoutError) as e:
                last = e
                self._final_url = None  # signed redirect URLs expire: resolve again
                time.sleep(2 ** attempt)
        raise IOError(f"range fetch {start}-{end} failed after {self.retries} attempts: {last}")

    # RawIOBase interface
    def readable(self):
        return True

    def seekable(self):
        return True

    def tell(self):
        return self.pos

    def seek(self, offset, whence=io.SEEK_SET):
        if whence == io.SEEK_SET:
            self.pos = offset
        elif whence == io.SEEK_CUR:
            self.pos += offset
        elif whence == io.SEEK_END:
            self.pos = self.size + offset
        else:
            raise ValueError(whence)
        if self.pos < 0:
            raise ValueError("negative seek position")
        return self.pos

    def readinto(self, b):
        n = min(len(b), self.size - self.pos)
        if n <= 0:
            return 0
        data = self._fetch(self.pos, self.pos + n - 1)
        b[: len(data)] = data
        self.pos += len(data)
        self.n_requests += 1
        self.bytes_fetched += len(data)
        return len(data)


def open_remote_zip(url: str, expected_bytes: int | None = None, buffer_size: int = 1 << 18):
    """Return (ZipFile, raw HTTPRangeFile) for a remote archive."""
    import zipfile

    raw = HTTPRangeFile(url)
    if expected_bytes is not None and raw.size != expected_bytes:
        raise IOError(f"remote size {raw.size} != recorded {expected_bytes}")
    zf = zipfile.ZipFile(io.BufferedReader(raw, buffer_size=buffer_size))
    return zf, raw


def find_member(names, suffix: str) -> str:
    hits = [n for n in names if n.endswith(suffix) and not n.split("/")[-1].startswith("._")]
    if len(hits) != 1:
        raise KeyError(f"expected exactly one member ending in {suffix!r}, found {hits[:5]}")
    return hits[0]


def file_digest(path: str, algo: str = "sha256", chunk: int = 1 << 22) -> str:
    h = hashlib.new(algo)
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()


# ---- metadata parsing ------------------------------------------------------------
def parse_asvspoof_cm_protocol(text: str) -> list[dict]:
    """ASVspoof 2019 LA CM protocol: SPEAKER_ID AUDIO_FILE_NAME - SYSTEM_ID KEY."""
    out = []
    for i, line in enumerate(text.splitlines(), 1):
        if not line.strip():
            continue
        parts = line.split()
        if len(parts) != 5:
            raise ValueError(f"protocol line {i}: expected 5 columns, got {len(parts)}: {line!r}")
        spk, utt, _unused, system, key = parts
        if key not in ("bonafide", "spoof"):
            raise ValueError(f"protocol line {i}: unknown key {key!r}")
        if key == "spoof" and system == "-":
            raise ValueError(f"protocol line {i}: spoof without attack ID")
        out.append({
            "clip_id": utt, "speaker_id": spk,
            "attack_id": system if key == "spoof" else "",
            "label": SYNTHETIC if key == "spoof" else GENUINE,
            "original_label": key,
        })
    return out


ITW_LABELS = {"bona-fide": GENUINE, "bonafide": GENUINE, "spoof": SYNTHETIC}


def parse_itw_meta(text: str) -> list[dict]:
    """In-the-Wild meta.csv (expected columns: file, speaker, label)."""
    rows = list(csv.DictReader(io.StringIO(text)))
    if not rows:
        raise ValueError("meta.csv is empty")
    cols = {c.strip().lower(): c for c in rows[0].keys()}
    missing = {"file", "speaker", "label"} - set(cols)
    if missing:
        raise ValueError(f"meta.csv lacks columns {missing}; has {list(rows[0].keys())}")
    out = []
    for i, r in enumerate(rows, 2):
        lab = r[cols["label"]].strip()
        if lab.lower() not in ITW_LABELS:
            raise ValueError(f"meta.csv line {i}: unknown label {lab!r}")
        fn = r[cols["file"]].strip()
        out.append({
            "clip_id": fn, "speaker_id": r[cols["speaker"]].strip(), "attack_id": "",
            "label": ITW_LABELS[lab.lower()], "original_label": lab,
        })
    return out


def describe_records(records: list[dict]) -> dict:
    by_label = Counter(r["label"] for r in records)
    attacks = Counter(r["attack_id"] for r in records if r["label"] == SYNTHETIC and r["attack_id"])
    spk = defaultdict(Counter)
    for r in records:
        spk[r["label"]][r["speaker_id"]] += 1
    return {
        "n": len(records),
        "by_label": dict(sorted(by_label.items())),
        "by_attack": dict(sorted(attacks.items())),
        "n_speakers_by_label": {k: len(v) for k, v in sorted(spk.items())},
        "n_speakers_total": len({r["speaker_id"] for r in records}),
        "duplicate_clip_ids": len(records) - len({r["clip_id"] for r in records}),
    }


# ---- score-free, reproducible selection ------------------------------------------
FORBIDDEN_SELECTION_KEYS = ("score", "logit", "pred", "prob")


def _rng(seed: int, *stratum) -> random.Random:
    # str seeds are hashed with SHA-512 by random.seed (version 2): stable across
    # runs, machines and PYTHONHASHSEED. One RNG per stratum => order-independent.
    return random.Random(":".join([str(seed), *map(str, stratum)]))


def speaker_round_robin(records: list[dict], n: int, seed: int, *stratum) -> list[dict]:
    """Take n clips spreading over speakers: shuffled speaker order, shuffled clips per
    speaker, then one clip per speaker per round. Uses metadata only."""
    by_spk = defaultdict(list)
    for r in sorted(records, key=lambda r: r["clip_id"]):
        by_spk[r["speaker_id"]].append(r)
    spks = sorted(by_spk)
    _rng(seed, *stratum, "speakers").shuffle(spks)
    for s in spks:
        _rng(seed, *stratum, "clips", s).shuffle(by_spk[s])
    out, rnd = [], 0
    while len(out) < n and any(len(by_spk[s]) > rnd for s in spks):
        for s in spks:
            if len(out) >= n:
                break
            if len(by_spk[s]) > rnd:
                out.append(by_spk[s][rnd])
        rnd += 1
    return out


def select_subset(records: list[dict], dataset: str, seed: int, n_genuine: int,
                  n_synthetic: int | None = None, n_per_attack: int | None = None,
                  full: bool = False, exclude_clip_ids=(), exclude_speakers=()) -> tuple[list[dict], dict]:
    """Score-free stratified sample. full=True: every record (sorted by ID).

    exclude_clip_ids / exclude_speakers: clips already used as development
    evidence (e.g. an earlier pilot) are removed BEFORE sampling, so a fresh
    held-out subset never shares clips (or speakers) with them.
    """
    for r in records[:1]:
        bad = [k for k in r if any(f in k.lower() for f in FORBIDDEN_SELECTION_KEYS)]
        assert not bad, f"selection input contains detector-derived fields: {bad}"
    ex_ids, ex_spk = set(exclude_clip_ids), set(exclude_speakers)
    n_before = len(records)
    records = [r for r in records if r["clip_id"] not in ex_ids and r["speaker_id"] not in ex_spk]
    exclusion = {"excluded_clip_ids": len(ex_ids), "excluded_speakers": len(ex_spk),
                 "records_before": n_before, "records_after": len(records)}
    gen = [r for r in records if r["label"] == GENUINE]
    syn = [r for r in records if r["label"] == SYNTHETIC]
    if full:
        sel = [dict(r, stratum="all") for r in sorted(records, key=lambda r: r["clip_id"])]
        rule = "full evaluation partition: every protocol/metadata row, sorted by clip_id"
    else:
        sel = [dict(r, stratum="genuine") for r in speaker_round_robin(gen, n_genuine, seed, dataset, GENUINE)]
        if n_per_attack is not None:
            by_atk = defaultdict(list)
            for r in syn:
                by_atk[r["attack_id"]].append(r)
            for atk in sorted(by_atk):
                take = speaker_round_robin(by_atk[atk], n_per_attack, seed, dataset, SYNTHETIC, atk)
                sel += [dict(r, stratum=f"synthetic/{atk}") for r in take]
            rule = (f"genuine: {n_genuine} by speaker round-robin; synthetic: {n_per_attack} per attack ID "
                    f"({len(by_atk)} attacks) by speaker round-robin within attack")
        else:
            sel += [dict(r, stratum="synthetic") for r in speaker_round_robin(syn, n_synthetic, seed, dataset, SYNTHETIC)]
            rule = f"genuine: {n_genuine}, synthetic: {n_synthetic}, each by speaker round-robin"
        rule += (f"; seed {seed}; per-stratum random.Random('{seed}:{dataset}:<label>[:<attack>]:...'); "
                 "input sorted by clip_id; metadata only (no audio, no detector output)")
    for i, r in enumerate(sel):
        r["selection_rank"] = i
        r["dataset"] = dataset
    ids = "\n".join(r["clip_id"] for r in sel)
    meta = {
        "dataset": dataset, "seed": seed, "full": full, "rule": rule,
        "n_selected": len(sel), "selected": describe_records(sel), "available": describe_records(records),
        "clip_id_list_sha256": hashlib.sha256(ids.encode()).hexdigest(),
        "shortfalls": {}, "exclusion": exclusion,
    }
    assert not ({r["clip_id"] for r in sel} & ex_ids) and not ({r["speaker_id"] for r in sel} & ex_spk)
    if not full:
        got = Counter(r["stratum"] for r in sel)
        if got["genuine"] < n_genuine:
            meta["shortfalls"]["genuine"] = f"{got['genuine']} < {n_genuine}"
        for k, v in got.items():
            if k.startswith("synthetic/") and n_per_attack and v < n_per_attack:
                meta["shortfalls"][k] = f"{v} < {n_per_attack}"
    return sel, meta


# ---- metrics (positive class = synthetic; higher score = more synthetic) --------
def _split(labels, scores):
    pos = [s for l, s in zip(labels, scores) if l == 1]
    neg = [s for l, s in zip(labels, scores) if l == 0]
    return pos, neg


def roc_auc(labels, scores):
    """Mann-Whitney U / (n_pos * n_neg), ties counted 1/2. None if a class is empty."""
    pos, neg = _split(labels, scores)
    if not pos or not neg:
        return None
    allv = sorted([(s, 1) for s in pos] + [(s, 0) for s in neg])
    rank_sum_pos, i = 0.0, 0
    while i < len(allv):
        j = i
        while j < len(allv) and allv[j][0] == allv[i][0]:
            j += 1
        avg_rank = (i + 1 + j) / 2.0  # ranks i+1..j
        rank_sum_pos += avg_rank * sum(1 for k in range(i, j) if allv[k][1] == 1)
        i = j
    u = rank_sum_pos - len(pos) * (len(pos) + 1) / 2.0
    return u / (len(pos) * len(neg))


def eer(labels, scores):
    """EER with the ASVspoof 2019 reference convention (compute_eer in the official
    tDCF package): sweep thresholds over all scores, take the point minimising
    |FNR - FPR| and report their mean. Decision: synthetic if score >= t.
    Returns (eer, threshold_at_eer) or (None, None). The EER threshold is
    descriptive only; it is NOT an operating point for this project."""
    pos, neg = _split(labels, scores)
    if not pos or not neg:
        return None, None
    pts = sorted(set(pos) | set(neg)) + [math.inf]  # inf: FNR = 1, FPR = 0
    pos_s, neg_s = sorted(pos), sorted(neg)
    best = None
    ip = ineg = 0
    for t in pts:  # FNR(t) = #pos < t / P ; FPR(t) = #neg >= t / N
        while ip < len(pos_s) and pos_s[ip] < t:
            ip += 1
        while ineg < len(neg_s) and neg_s[ineg] < t:
            ineg += 1
        fnr, fpr = ip / len(pos_s), (len(neg_s) - ineg) / len(neg_s)
        d = abs(fnr - fpr)
        if best is None or d < best[0]:
            best = (d, (fnr + fpr) / 2.0, t)
    return best[1], best[2]


def wilson(k: int, n: int, z: float = 1.959964):
    if n == 0:
        return None, None
    p = k / n
    den = 1 + z * z / n
    c = (p + z * z / (2 * n)) / den
    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / den
    return max(0.0, c - h), min(1.0, c + h)


def at_threshold(labels, scores, thr: float) -> dict:
    tp = sum(1 for l, s in zip(labels, scores) if l == 1 and s >= thr)
    fn = sum(1 for l, s in zip(labels, scores) if l == 1 and s < thr)
    fp = sum(1 for l, s in zip(labels, scores) if l == 0 and s >= thr)
    tn = sum(1 for l, s in zip(labels, scores) if l == 0 and s < thr)
    P, N = tp + fn, fp + tn
    rec = tp / P if P else None
    fpr = fp / N if N else None
    spec = tn / N if N else None
    prec = tp / (tp + fp) if tp + fp else None
    f1 = (2 * prec * rec / (prec + rec)) if prec is not None and rec is not None and (prec + rec) else None
    return {
        "threshold": thr,
        "confusion": {"tp_synthetic_as_synthetic": tp, "fn_synthetic_as_genuine": fn,
                      "fp_genuine_as_synthetic": fp, "tn_genuine_as_genuine": tn},
        "synthetic_precision": prec, "f1_synthetic": f1,
        "synthetic_recall": rec, "synthetic_recall_wilson95": wilson(tp, P),
        "genuine_fpr": fpr, "genuine_fpr_wilson95": wilson(fp, N),
        "genuine_specificity": spec,
        "balanced_accuracy": (rec + spec) / 2 if rec is not None and spec is not None else None,
        "accuracy": (tp + tn) / (P + N) if P + N else None,
        "n_synthetic": P, "n_genuine": N,
    }


def bootstrap_ci(labels, scores, thr: float, n_boot: int = 1000, seed: int = 0) -> dict:
    """Stratified (within-class) percentile bootstrap, 95 %. Reflects clip sampling
    only; it ignores speaker clustering, so intervals are optimistic."""
    pos, neg = _split(labels, scores)
    if not pos or not neg or n_boot <= 0:
        return {}
    rng = random.Random(f"bootstrap:{seed}")
    acc = defaultdict(list)
    for _ in range(n_boot):
        p = [pos[rng.randrange(len(pos))] for _ in pos]
        q = [neg[rng.randrange(len(neg))] for _ in neg]
        lab, sc = [1] * len(p) + [0] * len(q), p + q
        acc["roc_auc"].append(roc_auc(lab, sc))
        acc["eer"].append(eer(lab, sc)[0])
        acc["balanced_accuracy@thr"].append(at_threshold(lab, sc, thr)["balanced_accuracy"])
    out = {}
    for k, v in acc.items():
        v = sorted(v)
        out[k] = (v[int(0.025 * (len(v) - 1))], v[int(math.ceil(0.975 * (len(v) - 1)))])
    out["n_boot"], out["seed"] = n_boot, seed
    return out


def evaluate(labels, scores, thr: float, n_boot: int = 1000, seed: int = 0) -> dict:
    """labels: 1 = synthetic, 0 = genuine (scored clips only)."""
    e, et = eer(labels, scores)
    return {
        "n_scored": len(labels),
        "threshold_free": {"roc_auc": roc_auc(labels, scores), "eer": e,
                           "eer_threshold_descriptive_only": et},
        "at_threshold": at_threshold(labels, scores, thr),
        "bootstrap95": bootstrap_ci(labels, scores, thr, n_boot, seed),
    }


REJECT_CATEGORIES = (
    ("too short", "too_short"), ("mostly silent", "mostly_silent"), ("silent", "silent"),
    ("unreadable", "unreadable"), ("empty", "empty"), ("NaN", "non_finite"),
)


def reject_category(reason: str) -> str:
    for needle, cat in REJECT_CATEGORIES:
        if reason.startswith(needle) or needle in reason[:40]:
            return cat
    return "other"


def jdump(obj, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False, default=str)


# ---- robustness verdict (official ratio) ---------------------------------------------
ROBUSTNESS_RATIO_LIMIT = 2.5
REQUIRED_CONDITIONS = ("clean", "narrowband_8k", "codec_1", "codec_2", "noise")


def robustness_verdict(eers: dict, required=REQUIRED_CONDITIONS, limit: float = ROBUSTNESS_RATIO_LIMIT,
                       n_pairs: int | None = None) -> dict:
    """eers: {condition: EER or None} computed on the SAME paired clips.

    PASS  : every required condition present, clean EER > 0, worst degraded / clean <= limit.
    FAIL  : every required condition present, clean EER > 0, ratio > limit.
    INCOMPLETE: a required condition is missing or None.
    UNDEFINED_RATIO: clean EER == 0 (ratio undefined). The absolute degraded EERs are reported and, as an
        auxiliary number only, the ratio against a one-error floor 1/n_pairs (never used for PASS).
    """
    missing = [c for c in required if eers.get(c) is None]
    out = {"required": list(required), "missing": missing, "limit": limit, "eers": {c: eers.get(c) for c in required}}
    if missing:
        out.update(verdict="INCOMPLETE", ratio=None, worst_condition=None)
        return out
    clean = eers["clean"]
    degraded = {c: eers[c] for c in required if c != "clean"}
    worst = max(degraded, key=degraded.get)
    out["worst_condition"], out["worst_eer"] = worst, degraded[worst]
    out["eer_8k_minus_clean_pp"] = round((eers["narrowband_8k"] - clean) * 100, 3)
    if clean == 0:
        floor = (1.0 / n_pairs) if n_pairs else None
        out.update(verdict="UNDEFINED_RATIO", ratio=None,
                   auxiliary_ratio_vs_one_error_floor=(degraded[worst] / floor) if floor else None,
                   note="clean EER is 0: the ratio is undefined; see absolute degraded EERs")
        return out
    ratio = degraded[worst] / clean
    out.update(ratio=ratio, verdict="PASS" if ratio <= limit else "FAIL")
    return out


# ---- pre-registered denoiser-frontend selection (development evidence only) --------------
def frontend_selection(dev_eers: dict, baseline: str = "none", margin_pp: float = 1.0,
                       required=REQUIRED_CONDITIONS, order: list | None = None) -> dict:
    """dev_eers: {frontend: {condition: EER}} on the SAME paired development clips.

    Rule (fixed before looking at fresh data):
      1. a frontend is eligible only if its clean EER <= clean EER of `baseline` + margin_pp/100
         (it must not destroy synthesis artefacts on clean audio);
      2. among eligible frontends pick the lowest worst EER over the degraded required conditions;
      3. ties -> earlier in `order` (simplest first).
    Returns the frozen choice (None if nothing is eligible) and the full table.
    """
    degraded = [c for c in required if c != "clean"]
    base_clean = dev_eers[baseline]["clean"]
    table = {}
    for fe, e in dev_eers.items():
        if any(e.get(c) is None for c in required):
            table[fe] = {"eligible": False, "reason": "missing condition"}
            continue
        worst_c = max(degraded, key=lambda c: e[c])
        table[fe] = {"clean_eer": e["clean"], "clean_delta_pp": (e["clean"] - base_clean) * 100,
                     "worst_condition": worst_c, "worst_eer": e[worst_c],
                     "ratio_unrounded": (e[worst_c] / e["clean"]) if e["clean"] > 0 else None,
                     "eligible": fe != baseline and e["clean"] <= base_clean + margin_pp / 100}
    order = order or [fe for fe in dev_eers if fe != baseline]
    elig = [fe for fe in order if table.get(fe, {}).get("eligible")]
    chosen = min(elig, key=lambda fe: (table[fe]["worst_eer"], order.index(fe))) if elig else None
    improves = chosen is not None and table[chosen]["worst_eer"] < table[baseline]["worst_eer"]
    return {"frozen_frontend": chosen if improves else None, "candidate_if_not_improving": chosen if not improves else None,
            "margin_pp": margin_pp, "baseline": baseline, "table": table,
            "rule": "eligible if clean EER <= baseline clean + margin; then min worst degraded EER; ties -> simplest; "
                    "frozen only if it beats the baseline's worst EER"}


def clean_margin_check(clean_eer: float, baseline_clean_eer: float, margin_pp: float) -> dict:
    """Guards against a higher clean EER disguising a 'better' ratio."""
    delta = (clean_eer - baseline_clean_eer) * 100
    return {"clean_eer": clean_eer, "baseline_clean_eer": baseline_clean_eer, "delta_pp": delta,
            "margin_pp": margin_pp, "within_margin": delta <= margin_pp}

In [ ]:
%%writefile antideepfake_hf.py
"""Candidate detector: AntiDeepfake Wav2Vec2-Small (NII Yamagishi Lab), loaded WITHOUT fairseq.

Model: nii-yamagishilab/wav2vec-small-anti-deepfake @ MODEL_REVISION, model.safetensors
(380,210,632 B, sha256 MODEL_SHA256). Licence CC BY-NC-SA 4.0 (research/educational use;
NOT for commercial deployment without permission). Base: facebook/wav2vec2-base (Apache-2.0).
Paper: Ge, Wang, Liu, Yamagishi, "Post-training for Deepfake Speech Detection", arXiv:2506.21090.
We did not train or fine-tune this model.

Why this loader exists: the published inference code builds the encoder with
fairseq 0.12.2 (Python <= 3.8 wheels; the project is archived), which does not install on
current Colab runtimes. The checkpoint's encoder keys are fairseq wav2vec2 names, so we map
them onto transformers' Wav2Vec2Model using the same correspondence as HF's official
convert_wav2vec2_original_pytorch_checkpoint_to_pytorch.py, and FAIL LOUDLY if any
parameter is left over or missing. Numerical identity with the fairseq path is NOT
proven here (see verify_notes()); the notebook checks behaviour on development evidence
before any held-out scoring.

Architecture correspondence, checked against fairseq v0.12.2 fairseq/models/wav2vec/wav2vec2.py and
transformers modeling_wav2vec2.py (code read side by side; see architecture_check()):
  * conv feature extractor: 7 Conv1d(no bias) [(512,10,5),(512,3,2)x4,(512,2,2)x2]; fairseq
    extractor_mode='default' = GroupNorm(512,512) after conv 0 only, GELU after every conv
    == HF feat_extract_norm='group', conv_bias=False, feat_extract_activation='gelu'.
  * LayerNorm(512) on frames, then Linear(512,768) (fairseq model.layer_norm + post_extract_proj
    == HF feature_projection.layer_norm + .projection).
  * positional conv: Conv1d(768,768,k=128,pad=64,groups=16), weight_norm(dim=2), drop last frame
    (SamePad == Wav2Vec2SamePadLayer), GELU; x = x + pos.
  * post-LN transformer (fairseq layer_norm_first=False == HF do_stable_layer_norm=False):
    encoder.layer_norm BEFORE the 12 layers; each layer: attn -> residual -> layer_norm -> FFN(GELU,
    3072) -> residual -> final_layer_norm; 12 heads, scaling head_dim**-0.5; exact (erf) GELU both.
    No final norm after the last layer in either implementation. Output = last_hidden_state == fairseq 'x'.
  * fairseq pads frames to a multiple of 2 with a key-padding mask and removes the pad afterwards;
    with the pad masked this is a no-op for the result. HF does no such padding. No attention mask is
    used by either side for a single unpadded clip (batch size 1).
  * head: mean over time (AdaptiveAvgPool1d(1)) -> Linear(768,2) -> softmax; fake = index 0.
  * eval mode: no time/feature masking, no dropout, no layerdrop on either side.
  Differences that remain: the authors decode with torchaudio and resample with torchaudio.functional.resample
  (our path: soundfile/librosa + soxr; irrelevant at 16 kHz input), and they score whole files while we apply
  the 2-10 s policy (first 10 s). Numerical parity is only established by the optional fairseq parity cell.

Published recipe (model card), reproduced here:
  mono (mean) -> 16 kHz -> layer_norm over the whole waveform (zero mean, unit var, eps 1e-5)
  -> encoder last hidden state -> mean over time -> Linear(768, 2) -> softmax.
  Output order: index 0 = FAKE, index 1 = REAL (card: "<Fake score, Real score>").
  Arbitrary input length (no cropping in the published code).
"""

from __future__ import annotations

import time

try:
    import numpy as np
except ImportError:  # the key-mapping functions are stdlib-only and testable without numpy
    np = None

MODEL_REPO = "nii-yamagishilab/wav2vec-small-anti-deepfake"
MODEL_REVISION = "9a13264b5dcc827a8d5a4f8e01fccefa392f886b"
MODEL_FILE = "model.safetensors"
MODEL_SHA256 = "828ee456122f86d5d631cb7895a10e5c62c78a4fcb8a8b1c42cb5838a9abcfe0"
MODEL_BYTES = 380210632
LICENSE = "CC-BY-NC-SA-4.0 (weights); base facebook/wav2vec2-base Apache-2.0; code BSD-3"
FAKE_INDEX = 0          # softmax index of the synthetic class (card: <Fake, Real>)
# The card reports EER 4.24 % "@ 0.8128" on In-the-Wild. That threshold is the EER point found ON In-the-Wild
# with the authors' whole-file pipeline, i.e. it was selected on the dataset we test on. It is therefore NOT a
# pre-registered operating point for us and is reported only as context, never as a decision threshold.
CARD_EER_THRESHOLD_ITW_CONTEXT_ONLY = 0.8128
SR = 16000

# Keys of the fairseq checkpoint that belong to pre-training only (quantiser, projections,
# mask embedding). They have no counterpart in inference and are dropped on purpose.
PRETRAIN_ONLY_PREFIXES = ("quantizer.", "project_q.", "final_proj.", "mask_emb")


def _map_encoder_key(k: str) -> str | None:
    """fairseq Wav2Vec2Model key (prefix 'm_ssl.model.' removed) -> transformers Wav2Vec2Model key."""
    if k.startswith(PRETRAIN_ONLY_PREFIXES):
        return None
    if k.startswith("feature_extractor.conv_layers."):
        _, _, layer, typ, param = k.split(".")
        if typ == "0":
            return f"feature_extractor.conv_layers.{layer}.conv.{param}"
        if typ == "2":  # group norm (layer 0 only, feat_extract_norm='group')
            return f"feature_extractor.conv_layers.{layer}.layer_norm.{param}"
        raise KeyError(k)
    if k.startswith("post_extract_proj."):
        return "feature_projection.projection." + k.split(".", 1)[1]
    if k.startswith("layer_norm."):
        return "feature_projection.layer_norm." + k.split(".", 1)[1]
    if k.startswith("encoder.pos_conv.0."):
        param = k.rsplit(".", 1)[1]           # weight_g / weight_v / bias
        return "encoder.pos_conv_embed.conv." + param
    if k.startswith("encoder.layer_norm."):
        return k
    if k.startswith("encoder.layers."):
        _, _, i, rest = k.split(".", 3)
        sub = {
            "self_attn.k_proj": "attention.k_proj", "self_attn.v_proj": "attention.v_proj",
            "self_attn.q_proj": "attention.q_proj", "self_attn.out_proj": "attention.out_proj",
            "self_attn_layer_norm": "layer_norm", "fc1": "feed_forward.intermediate_dense",
            "fc2": "feed_forward.output_dense", "final_layer_norm": "final_layer_norm",
        }
        for src, dst in sub.items():
            if rest.startswith(src + "."):
                return f"encoder.layers.{i}.{dst}.{rest[len(src) + 1:]}"
        raise KeyError(k)
    raise KeyError(k)


def convert_state_dict(sd: dict, hf_keys: set) -> tuple[dict, dict, dict]:
    """Return (encoder_state_dict, head_state_dict, report). Raises on anything unexplained."""
    enc, head, dropped, unexplained = {}, {}, [], []
    for k, v in sd.items():
        if k.startswith("proj_fc."):
            head[k.split(".", 1)[1]] = v
            continue
        if not k.startswith("m_ssl.model."):
            unexplained.append(k)
            continue
        kk = k[len("m_ssl.model."):]
        try:
            mk = _map_encoder_key(kk)
        except KeyError:
            unexplained.append(k)
            continue
        if mk is None:
            dropped.append(k)
            continue
        # weight-norm naming differs across torch/transformers versions
        if mk.endswith(("weight_g", "weight_v")) and mk not in hf_keys:
            alt = mk.replace("weight_g", "parametrizations.weight.original0").replace(
                "weight_v", "parametrizations.weight.original1")
            if alt in hf_keys:
                mk = alt
        enc[mk] = v
    missing = sorted(hf_keys - set(enc))
    unexpected = sorted(set(enc) - hf_keys)
    report = {"n_checkpoint_tensors": len(sd), "n_encoder_mapped": len(enc), "n_head": len(head),
              "dropped_pretrain_only": dropped, "unexplained": unexplained,
              "missing_in_checkpoint": missing, "unexpected_for_hf": unexpected}
    if unexplained or missing or unexpected or set(head) != {"weight", "bias"}:
        raise RuntimeError(f"checkpoint/model mismatch: {report}")
    return enc, head, report


# fairseq Wav2Vec2Config defaults that the authors' code relies on (only 5 fields are set explicitly there)
EXPECTED_HF_CONFIG = {
    "model_type": "wav2vec2", "feat_extract_norm": "group", "conv_bias": False, "feat_extract_activation": "gelu",
    "conv_dim": [512] * 7, "conv_kernel": [10, 3, 3, 3, 3, 2, 2], "conv_stride": [5, 2, 2, 2, 2, 2, 2],
    "hidden_size": 768, "num_hidden_layers": 12, "num_attention_heads": 12, "intermediate_size": 3072,
    "hidden_act": "gelu", "do_stable_layer_norm": False, "num_conv_pos_embeddings": 128,
    "num_conv_pos_embedding_groups": 16, "layer_norm_eps": 1e-05, "add_adapter": False,
}


def architecture_check(cfg) -> dict:
    """Compare the HF config against the fairseq-default architecture the checkpoint was trained with."""
    got = {k: getattr(cfg, k) for k in EXPECTED_HF_CONFIG}
    bad = {k: (got[k], v) for k, v in EXPECTED_HF_CONFIG.items() if got[k] != v}
    if bad:
        raise RuntimeError(f"HF config does not match the fairseq architecture: {bad}")
    return got


class AntiDeepfakeHF:
    """Thin inference wrapper (torch + transformers). Build with .load()."""

    def __init__(self, encoder, head, device, report):
        self.encoder, self.head, self.device, self.report = encoder, head, device, report

    @classmethod
    def load(cls, safetensors_path: str, config_source: str, device: str = "cpu"):
        import torch
        from safetensors.torch import load_file
        from transformers import Wav2Vec2Config, Wav2Vec2Model

        cfg = Wav2Vec2Config.from_pretrained(config_source)
        arch = architecture_check(cfg)
        # inference only: no masking, no layerdrop, no dropout
        cfg.mask_time_prob = 0.0
        cfg.mask_feature_prob = 0.0
        cfg.layerdrop = 0.0
        cfg.apply_spec_augment = False
        encoder = Wav2Vec2Model(cfg)
        sd = load_file(safetensors_path)
        hf_keys = set(encoder.state_dict().keys()) - {"masked_spec_embed"}
        enc_sd, head_sd, report = convert_state_dict(sd, hf_keys)
        res = encoder.load_state_dict(enc_sd, strict=False)
        assert not res.unexpected_keys and set(res.missing_keys) <= {"masked_spec_embed"}, res
        head = torch.nn.Linear(768, 2)
        head.load_state_dict(head_sd)
        encoder.eval().to(device)
        head.eval().to(device)
        report["architecture"] = arch
        report["n_params_encoder"] = sum(p.numel() for p in encoder.parameters())
        report["n_params_head"] = sum(p.numel() for p in head.parameters())
        return cls(encoder, head, device, report)

    @staticmethod
    def preprocess(x16: np.ndarray) -> np.ndarray:
        """Published recipe: layer_norm over the whole waveform (eps 1e-5)."""
        x = x16.astype(np.float64)
        return ((x - x.mean()) / np.sqrt(x.var() + 1e-5)).astype(np.float32)

    def logits(self, wave: np.ndarray) -> np.ndarray:
        """wave: preprocessed mono 16 kHz float32 [n]. Returns logits [2] (index 0 = fake)."""
        import torch

        with torch.no_grad():
            t = torch.from_numpy(wave)[None].to(self.device)
            h = self.encoder(t).last_hidden_state          # [1, T, 768]
            z = self.head(h.mean(dim=1))                    # [1, 2]
        return z[0].float().cpu().numpy()


def softmax(z):
    z = np.asarray(z, dtype=np.float64)
    z = z - z.max()
    e = np.exp(z)
    return e / e.sum()


def score_clip(model: AntiDeepfakeHF, x16: np.ndarray, window=None, hop=None) -> dict:
    """Primary: whole analysed clip in one pass (as published). If window/hop are given,
    additionally scores 4 s windows (mean of fake scores) for comparability with Dhwani."""
    t0 = time.perf_counter()
    x = model.preprocess(x16)
    z = model.logits(x)
    p = softmax(z)
    out = {"logits_fake_real": [round(float(a), 4) for a in z], "fake_score_whole": round(float(p[FAKE_INDEX]), 6),
           "n_samples": int(len(x16))}
    if window:
        n = len(x)
        starts = [0] if n <= window else list(range(0, n - window + 1, hop))
        if n > window and starts[-1] + window < n:
            starts.append(n - window)
        ws = []
        for s in starts:
            w = x[s:s + window]
            if len(w) < window:
                w = np.pad(w, (0, window - len(w)))
            ws.append(float(softmax(model.logits(w))[FAKE_INDEX]))
        out.update(window_starts_s=[round(s / SR, 3) for s in starts], window_fake_scores=[round(v, 6) for v in ws],
                   fake_score_window_mean=round(float(np.mean(ws)), 6))
    out["latency_ms_model"] = round((time.perf_counter() - t0) * 1000, 1)
    return out


def verify_notes() -> str:
    return ("Weights were loaded through a fairseq->transformers key mapping (every checkpoint tensor accounted "
            "for) and the architecture fields were checked against fairseq defaults. This does NOT prove numerical "
            "parity with the authors' fairseq code; parity is 'unverified' unless the fairseq parity cell ran "
            "and reported a small max difference. The development-evidence gate (pilot clips) is a stop-and-"
            "investigate check: a low AUC does not by itself prove the mapping is wrong, and a high AUC does not "
            "prove parity.")

In [ ]:
%%writefile degrade.py
"""Paired audio degradations for the robustness evaluation (evaluation only).

Every function takes a mono 16 kHz float32 waveform (already decoded and validated by
dhwani_infer) and returns a mono 16 kHz float32 waveform of the same clip, so the comparison
between conditions is paired per clip. Codecs go through ffmpeg (Colab ships it); their exact
settings are fixed here and recorded in CONDITIONS. Noise is white Gaussian at a documented
SNR with a per-clip seed derived from (global seed, clip_id), so it is reproducible.

Required ("official ratio") conditions:  clean, narrowband_8k, codec_1 (Opus 16 kb/s),
codec_2 (MP3 32 kb/s), noise (white, 10 dB SNR).
Sensitivity checks (reported separately): loud_minus20db, loud_peak_minus1db, silence_pad_1s,
duration_first3s, telephone_quality (300-3400 Hz band-pass + 8-bit mu-law).
"""

from __future__ import annotations

import hashlib
import os
import shutil
import subprocess
import tempfile

import numpy as np

SR = 16000

CONDITIONS = {
    "clean": {"group": "required", "desc": "decoded 16 kHz mono, no change"},
    "narrowband_8k": {"group": "required", "desc": "resample 16 kHz -> 8 kHz -> 16 kHz, librosa soxr_hq both ways"},
    "codec_1": {"group": "required", "desc": "Opus (libopus) 16 kb/s VBR, application=voip, 16 kHz in/out via ffmpeg",
                "ffmpeg_encode": ["-c:a", "libopus", "-b:a", "16k", "-vbr", "on", "-application", "voip"], "ext": ".opus"},
    "codec_2": {"group": "required", "desc": "MP3 (libmp3lame) 32 kb/s CBR mono, 16 kHz via ffmpeg",
                "ffmpeg_encode": ["-c:a", "libmp3lame", "-b:a", "32k"], "ext": ".mp3"},
    "noise": {"group": "required", "desc": "additive white Gaussian noise at 10 dB SNR (SNR vs clip RMS), seed sha256(seed:clip_id)",
              "snr_db": 10.0},
    "loud_minus20db": {"group": "sensitivity", "desc": "gain -20 dB (quiet recording)"},
    "loud_peak_minus1db": {"group": "sensitivity", "desc": "peak-normalise to -1 dBFS (loud recording, no clipping)"},
    "silence_pad_1s": {"group": "sensitivity", "desc": "1.0 s of digital silence prepended and appended"},
    "duration_first3s": {"group": "sensitivity", "desc": "first 3.0 s only (clips shorter than 3 s unchanged, flagged)"},
    "telephone_quality": {"group": "sensitivity", "desc": "300-3400 Hz 4th-order Butterworth band-pass + 8-bit mu-law quantisation"},
}
NOISE_SEED = 20261027


def _rng(clip_id: str) -> np.random.Generator:
    h = hashlib.sha256(f"{NOISE_SEED}:{clip_id}".encode()).digest()
    return np.random.default_rng(int.from_bytes(h[:8], "little"))


def narrowband_8k(x: np.ndarray, **_) -> np.ndarray:
    import librosa

    y = librosa.resample(x, orig_sr=SR, target_sr=8000, res_type="soxr_hq")
    y = librosa.resample(y, orig_sr=8000, target_sr=SR, res_type="soxr_hq")
    return _fit(y, len(x))


def ffmpeg_available() -> dict:
    """Return {'ffmpeg': path or None, 'libopus': bool, 'libmp3lame': bool}."""
    exe = shutil.which("ffmpeg")
    out = {"ffmpeg": exe, "libopus": False, "libmp3lame": False}
    if exe:
        enc = subprocess.run([exe, "-hide_banner", "-encoders"], capture_output=True, text=True).stdout
        out["libopus"], out["libmp3lame"] = "libopus" in enc, "libmp3lame" in enc
    return out


def codec_roundtrip(x: np.ndarray, cond: str, **_) -> np.ndarray:
    import soundfile as sf

    spec = CONDITIONS[cond]
    d = tempfile.mkdtemp(prefix="vcd_codec_")
    try:
        src, enc, dec = os.path.join(d, "in.wav"), os.path.join(d, "enc" + spec["ext"]), os.path.join(d, "out.wav")
        sf.write(src, x, SR, subtype="PCM_16")
        for cmd in ([ "ffmpeg", "-hide_banner", "-loglevel", "error", "-y", "-i", src, *spec["ffmpeg_encode"], "-ar", "16000", "-ac", "1", enc],
                    ["ffmpeg", "-hide_banner", "-loglevel", "error", "-y", "-i", enc, "-ar", "16000", "-ac", "1", "-c:a", "pcm_f32le", dec]):
            r = subprocess.run(cmd, capture_output=True, text=True, timeout=120)
            if r.returncode != 0:
                raise RuntimeError(f"ffmpeg failed ({cond}): {r.stderr.strip()[-300:]}")
        y, sr = sf.read(dec, dtype="float32", always_2d=True)
        assert sr == SR
        y = y.mean(axis=1)
    finally:
        shutil.rmtree(d, ignore_errors=True)
    # codecs add a few ms of delay/padding: align length to the source (crop/pad at the end) so the clip
    # stays paired; the encoder delay itself is part of what a real transmission would do.
    return _fit(y, len(x))


def add_noise(x: np.ndarray, clip_id: str = "", snr_db: float | None = None, **_) -> np.ndarray:
    snr_db = CONDITIONS["noise"]["snr_db"] if snr_db is None else snr_db
    rms = float(np.sqrt(np.mean(x.astype(np.float64) ** 2))) or 1e-6
    noise = _rng(clip_id).standard_normal(len(x)).astype(np.float32)
    noise *= rms / (10 ** (snr_db / 20)) / (float(np.sqrt(np.mean(noise.astype(np.float64) ** 2))) or 1e-6)
    return (x + noise).astype(np.float32)


def loud_minus20db(x: np.ndarray, **_) -> np.ndarray:
    return (x * 10 ** (-20 / 20)).astype(np.float32)


def loud_peak_minus1db(x: np.ndarray, **_) -> np.ndarray:
    peak = float(np.max(np.abs(x))) or 1e-6
    return (x * (10 ** (-1 / 20) / peak)).astype(np.float32)


def silence_pad_1s(x: np.ndarray, **_) -> np.ndarray:
    z = np.zeros(SR, dtype=np.float32)
    return np.concatenate([z, x, z])


def duration_first3s(x: np.ndarray, **_) -> np.ndarray:
    return x[: 3 * SR].astype(np.float32)


def telephone_quality(x: np.ndarray, **_) -> np.ndarray:
    from scipy.signal import butter, sosfiltfilt

    sos = butter(4, [300, 3400], btype="bandpass", fs=SR, output="sos")
    y = sosfiltfilt(sos, x.astype(np.float64))
    peak = float(np.max(np.abs(y))) or 1e-6
    y = y / peak * 0.9                       # avoid clipping before quantisation
    mu = 255.0
    comp = np.sign(y) * np.log1p(mu * np.abs(y)) / np.log1p(mu)
    q = np.round(comp * 127) / 127           # 8-bit mu-law levels
    y = np.sign(q) * (np.expm1(np.abs(q) * np.log1p(mu))) / mu
    return (y * peak / 0.9).astype(np.float32)


def _fit(y: np.ndarray, n: int) -> np.ndarray:
    y = np.asarray(y, dtype=np.float32)
    if len(y) >= n:
        return y[:n]
    return np.pad(y, (0, n - len(y)))


APPLY = {
    "clean": lambda x, **k: x.astype(np.float32),
    "narrowband_8k": narrowband_8k,
    "codec_1": lambda x, **k: codec_roundtrip(x, "codec_1"),
    "codec_2": lambda x, **k: codec_roundtrip(x, "codec_2"),
    "noise": add_noise,
    "loud_minus20db": loud_minus20db,
    "loud_peak_minus1db": loud_peak_minus1db,
    "silence_pad_1s": silence_pad_1s,
    "duration_first3s": duration_first3s,
    "telephone_quality": telephone_quality,
}


def apply(cond: str, x: np.ndarray, clip_id: str = "") -> np.ndarray:
    return APPLY[cond](x, clip_id=clip_id)

In [ ]:
%%writefile frontend.py
"""Lightweight denoising front-ends under test (experiment, not an assumed fix).

Each front-end maps a mono 16 kHz float32 waveform to a mono 16 kHz float32 waveform of the
same length. In the experiment the SAME front-end is applied to every input, genuine and
synthetic, clean and degraded, exactly as it would run in front of the detector on a phone.
Settings are fixed here (no per-clip or per-condition tuning) and recorded in FRONTENDS.

  none            : identity (baseline)
  nr_stationary   : noisereduce spectral gating, stationary profile, prop_decrease 0.8,
                    n_fft 512 / hop 128 (32 ms / 8 ms at 16 kHz), 1.5 std threshold. Pure DSP,
                    CPU, no weights -> portable to Android (Kotlin re-implementation possible).
  deepfilternet3  : DeepFilterNet3 (Schroeter et al.), ~2 M params, MIT/Apache, default
                    post-filter off, attenuation limit default (100 dB). Runs at 48 kHz internally
                    (resampled in/out). Has an ONNX/LADSPA deployment story. Optional: only if
                    `pip install deepfilternet` works in the runtime.
"""

from __future__ import annotations

import numpy as np

SR = 16000

FRONTENDS = {
    "none": {"desc": "identity", "params": 0},
    "nr_stationary": {"desc": "noisereduce spectral gating (stationary=True, prop_decrease=0.8, n_fft=512, hop_length=128, "
                              "n_std_thresh_stationary=1.5)", "params": 0,
                      "kwargs": dict(stationary=True, prop_decrease=0.8, n_fft=512, hop_length=128, n_std_thresh_stationary=1.5)},
    "deepfilternet3": {"desc": "DeepFilterNet3 default model via `df.enhance`, 48 kHz internal, post-filter off", "params": 2_000_000},
}

_DF = {}


def available() -> dict:
    out = {"none": True}
    try:
        import noisereduce  # noqa: F401
        out["nr_stationary"] = True
    except Exception as e:
        out["nr_stationary"] = f"unavailable: {type(e).__name__}"
    try:
        from df.enhance import init_df  # noqa: F401
        out["deepfilternet3"] = True
    except Exception as e:
        out["deepfilternet3"] = f"unavailable: {type(e).__name__}: {str(e)[:120]}"
    return out


def _fit(y, n):
    y = np.asarray(y, dtype=np.float32)
    return y[:n] if len(y) >= n else np.pad(y, (0, n - len(y)))


def nr_stationary(x: np.ndarray) -> np.ndarray:
    import noisereduce as nr

    y = nr.reduce_noise(y=x.astype(np.float32), sr=SR, **FRONTENDS["nr_stationary"]["kwargs"])
    return _fit(y, len(x))


def deepfilternet3(x: np.ndarray) -> np.ndarray:
    import torch
    from df.enhance import enhance, init_df
    import librosa

    if "model" not in _DF:
        model, df_state, _ = init_df(post_filter=False)
        _DF.update(model=model, state=df_state, sr=df_state.sr())
    y48 = librosa.resample(x.astype(np.float32), orig_sr=SR, target_sr=_DF["sr"], res_type="soxr_hq")
    with torch.no_grad():
        out = enhance(_DF["model"], _DF["state"], torch.from_numpy(y48)[None])
    y = librosa.resample(out[0].cpu().numpy(), orig_sr=_DF["sr"], target_sr=SR, res_type="soxr_hq")
    return _fit(y, len(x))


APPLY = {"none": lambda x: x.astype(np.float32), "nr_stationary": nr_stationary, "deepfilternet3": deepfilternet3}


def apply(name: str, x: np.ndarray) -> np.ndarray:
    return APPLY[name](x)

In [ ]:
%%writefile manifest_dev_heldout_in_the_wild.csv
clip_id,speaker_id,attack_id,label,original_label,archive_member,stratum,selection_rank,dataset,split
8119.wav,Malcolm X,,genuine,bona-fide,release_in_the_wild/8119.wav,genuine,0,in_the_wild,heldout
30700.wav,Dave Chappelle,,genuine,bona-fide,release_in_the_wild/30700.wav,genuine,1,in_the_wild,heldout
4292.wav,Calvin Coolidge,,genuine,bona-fide,release_in_the_wild/4292.wav,genuine,2,in_the_wild,heldout
21123.wav,Donald Trump,,genuine,bona-fide,release_in_the_wild/21123.wav,genuine,3,in_the_wild,heldout
21657.wav,2Pac,,genuine,bona-fide,release_in_the_wild/21657.wav,genuine,4,in_the_wild,heldout
27364.wav,Ronald Reagan,,genuine,bona-fide,release_in_the_wild/27364.wav,genuine,5,in_the_wild,heldout
9240.wav,Mark Zuckerberg,,genuine,bona-fide,release_in_the_wild/9240.wav,genuine,6,in_the_wild,heldout
7894.wav,Ayn Rand,,genuine,bona-fide,release_in_the_wild/7894.wav,genuine,7,in_the_wild,heldout
9502.wav,Alec Guinness,,genuine,bona-fide,release_in_the_wild/9502.wav,genuine,8,in_the_wild,heldout
3035.wav,Jeff Goldblum,,genuine,bona-fide,release_in_the_wild/3035.wav,genuine,9,in_the_wild,heldout
19027.wav,Queen Elizabeth II,,genuine,bona-fide,release_in_the_wild/19027.wav,genuine,10,in_the_wild,heldout
19803.wav,Bob Ross,,genuine,bona-fide,release_in_the_wild/19803.wav,genuine,11,in_the_wild,heldout
26719.wav,Lyndon Johnson,,genuine,bona-fide,release_in_the_wild/26719.wav,genuine,12,in_the_wild,heldout
15408.wav,Christopher Hitchens,,genuine,bona-fide,release_in_the_wild/15408.wav,genuine,13,in_the_wild,heldout
21329.wav,Martin Luther King,,genuine,bona-fide,release_in_the_wild/21329.wav,genuine,14,in_the_wild,heldout
25138.wav,Richard Nixon,,genuine,bona-fide,release_in_the_wild/25138.wav,genuine,15,in_the_wild,heldout
3717.wav,Gilbert Gottfried,,genuine,bona-fide,release_in_the_wild/3717.wav,genuine,16,in_the_wild,heldout
24719.wav,Nelson Mandela,,genuine,bona-fide,release_in_the_wild/24719.wav,genuine,17,in_the_wild,heldout
28508.wav,John Cleese,,genuine,bona-fide,release_in_the_wild/28508.wav,genuine,18,in_the_wild,heldout
31222.wav,Adam Driver,,genuine,bona-fide,release_in_the_wild/31222.wav,genuine,19,in_the_wild,heldout
29081.wav,JFK,,genuine,bona-fide,release_in_the_wild/29081.wav,genuine,20,in_the_wild,heldout
29865.wav,FDR,,genuine,bona-fide,release_in_the_wild/29865.wav,genuine,21,in_the_wild,heldout
25061.wav,George W. Bush,,genuine,bona-fide,release_in_the_wild/25061.wav,genuine,22,in_the_wild,heldout
9157.wav,Scarlett Johansson,,genuine,bona-fide,release_in_the_wild/9157.wav,genuine,23,in_the_wild,heldout
19265.wav,The Notorious B.I.G.,,genuine,bona-fide,release_in_the_wild/19265.wav,genuine,24,in_the_wild,heldout
29975.wav,Louis C.K.,,genuine,bona-fide,release_in_the_wild/29975.wav,genuine,25,in_the_wild,heldout
28164.wav,Harry Truman,,genuine,bona-fide,release_in_the_wild/28164.wav,genuine,26,in_the_wild,heldout
21255.wav,Billie Eilish,,genuine,bona-fide,release_in_the_wild/21255.wav,genuine,27,in_the_wild,heldout
22936.wav,Alan Watts,,genuine,bona-fide,release_in_the_wild/22936.wav,genuine,28,in_the_wild,heldout
17861.wav,Kanye West,,genuine,bona-fide,release_in_the_wild/17861.wav,genuine,29,in_the_wild,heldout
29854.wav,Tucker Carlson,,genuine,bona-fide,release_in_the_wild/29854.wav,genuine,30,in_the_wild,heldout
8504.wav,Orson Welles,,genuine,bona-fide,release_in_the_wild/8504.wav,genuine,31,in_the_wild,heldout
4707.wav,Bernie Sanders,,genuine,bona-fide,release_in_the_wild/4707.wav,genuine,32,in_the_wild,heldout
16803.wav,Milton Friedman,,genuine,bona-fide,release_in_the_wild/16803.wav,genuine,33,in_the_wild,heldout
25912.wav,Frank Sinatra,,genuine,bona-fide,release_in_the_wild/25912.wav,genuine,34,in_the_wild,heldout
25781.wav,Robert Kardashian,,genuine,bona-fide,release_in_the_wild/25781.wav,genuine,35,in_the_wild,heldout
21100.wav,Barack Obama,,genuine,bona-fide,release_in_the_wild/21100.wav,genuine,36,in_the_wild,heldout
12789.wav,Alexandria Ocasio-Cortez,,genuine,bona-fide,release_in_the_wild/12789.wav,genuine,37,in_the_wild,heldout
27119.wav,Arnold Schwarzenegger,,genuine,bona-fide,release_in_the_wild/27119.wav,genuine,38,in_the_wild,heldout
5483.wav,Nick Offerman,,genuine,bona-fide,release_in_the_wild/5483.wav,genuine,39,in_the_wild,heldout
7198.wav,Norm MacDonald,,genuine,bona-fide,release_in_the_wild/7198.wav,genuine,40,in_the_wild,heldout
21298.wav,Bill Burr,,genuine,bona-fide,release_in_the_wild/21298.wav,genuine,41,in_the_wild,heldout
19242.wav,Kamala Harris,,genuine,bona-fide,release_in_the_wild/19242.wav,genuine,42,in_the_wild,heldout
24427.wav,George Carlin,,genuine,bona-fide,release_in_the_wild/24427.wav,genuine,43,in_the_wild,heldout
5452.wav,Jimmy Carter,,genuine,bona-fide,release_in_the_wild/5452.wav,genuine,44,in_the_wild,heldout
20703.wav,Mitch Hedberg,,genuine,bona-fide,release_in_the_wild/20703.wav,genuine,45,in_the_wild,heldout
1747.wav,Louis Farrakhan,,genuine,bona-fide,release_in_the_wild/1747.wav,genuine,46,in_the_wild,heldout
3545.wav,Boris Johnson,,genuine,bona-fide,release_in_the_wild/3545.wav,genuine,47,in_the_wild,heldout
7948.wav,Winston Churchill,,genuine,bona-fide,release_in_the_wild/7948.wav,genuine,48,in_the_wild,heldout
6147.wav,Jerry Seinfeld,,genuine,bona-fide,release_in_the_wild/6147.wav,genuine,49,in_the_wild,heldout
9532.wav,William F. Buckley,,genuine,bona-fide,release_in_the_wild/9532.wav,genuine,50,in_the_wild,heldout
9375.wav,Dwight Eisenhower,,genuine,bona-fide,release_in_the_wild/9375.wav,genuine,51,in_the_wild,heldout
24982.wav,Bill Clinton,,genuine,bona-fide,release_in_the_wild/24982.wav,genuine,52,in_the_wild,heldout
5309.wav,Mr. Rogers,,genuine,bona-fide,release_in_the_wild/5309.wav,genuine,53,in_the_wild,heldout
10023.wav,Malcolm X,,genuine,bona-fide,release_in_the_wild/10023.wav,genuine,54,in_the_wild,heldout
26765.wav,Dave Chappelle,,genuine,bona-fide,release_in_the_wild/26765.wav,genuine,55,in_the_wild,heldout
2197.wav,Calvin Coolidge,,genuine,bona-fide,release_in_the_wild/2197.wav,genuine,56,in_the_wild,heldout
23566.wav,Donald Trump,,genuine,bona-fide,release_in_the_wild/23566.wav,genuine,57,in_the_wild,heldout
14691.wav,2Pac,,genuine,bona-fide,release_in_the_wild/14691.wav,genuine,58,in_the_wild,heldout
612.wav,Ronald Reagan,,genuine,bona-fide,release_in_the_wild/612.wav,genuine,59,in_the_wild,heldout
16827.wav,Mark Zuckerberg,,genuine,bona-fide,release_in_the_wild/16827.wav,genuine,60,in_the_wild,heldout
30246.wav,Ayn Rand,,genuine,bona-fide,release_in_the_wild/30246.wav,genuine,61,in_the_wild,heldout
25160.wav,Alec Guinness,,genuine,bona-fide,release_in_the_wild/25160.wav,genuine,62,in_the_wild,heldout
6291.wav,Jeff Goldblum,,genuine,bona-fide,release_in_the_wild/6291.wav,genuine,63,in_the_wild,heldout
6649.wav,Queen Elizabeth II,,genuine,bona-fide,release_in_the_wild/6649.wav,genuine,64,in_the_wild,heldout
6499.wav,Bob Ross,,genuine,bona-fide,release_in_the_wild/6499.wav,genuine,65,in_the_wild,heldout
4615.wav,Lyndon Johnson,,genuine,bona-fide,release_in_the_wild/4615.wav,genuine,66,in_the_wild,heldout
25954.wav,Christopher Hitchens,,genuine,bona-fide,release_in_the_wild/25954.wav,genuine,67,in_the_wild,heldout
8604.wav,Martin Luther King,,genuine,bona-fide,release_in_the_wild/8604.wav,genuine,68,in_the_wild,heldout
16681.wav,Richard Nixon,,genuine,bona-fide,release_in_the_wild/16681.wav,genuine,69,in_the_wild,heldout
15195.wav,Gilbert Gottfried,,genuine,bona-fide,release_in_the_wild/15195.wav,genuine,70,in_the_wild,heldout
5044.wav,Nelson Mandela,,genuine,bona-fide,release_in_the_wild/5044.wav,genuine,71,in_the_wild,heldout
5893.wav,John Cleese,,genuine,bona-fide,release_in_the_wild/5893.wav,genuine,72,in_the_wild,heldout
26640.wav,Adam Driver,,genuine,bona-fide,release_in_the_wild/26640.wav,genuine,73,in_the_wild,heldout
24780.wav,JFK,,genuine,bona-fide,release_in_the_wild/24780.wav,genuine,74,in_the_wild,heldout
29767.wav,FDR,,genuine,bona-fide,release_in_the_wild/29767.wav,genuine,75,in_the_wild,heldout
28755.wav,George W. Bush,,genuine,bona-fide,release_in_the_wild/28755.wav,genuine,76,in_the_wild,heldout
31284.wav,Scarlett Johansson,,genuine,bona-fide,release_in_the_wild/31284.wav,genuine,77,in_the_wild,heldout
4852.wav,The Notorious B.I.G.,,genuine,bona-fide,release_in_the_wild/4852.wav,genuine,78,in_the_wild,heldout
21419.wav,Louis C.K.,,genuine,bona-fide,release_in_the_wild/21419.wav,genuine,79,in_the_wild,heldout
27526.wav,Harry Truman,,genuine,bona-fide,release_in_the_wild/27526.wav,genuine,80,in_the_wild,heldout
18011.wav,Billie Eilish,,genuine,bona-fide,release_in_the_wild/18011.wav,genuine,81,in_the_wild,heldout
5758.wav,Alan Watts,,genuine,bona-fide,release_in_the_wild/5758.wav,genuine,82,in_the_wild,heldout
6817.wav,Kanye West,,genuine,bona-fide,release_in_the_wild/6817.wav,genuine,83,in_the_wild,heldout
29328.wav,Tucker Carlson,,genuine,bona-fide,release_in_the_wild/29328.wav,genuine,84,in_the_wild,heldout
26924.wav,Orson Welles,,genuine,bona-fide,release_in_the_wild/26924.wav,genuine,85,in_the_wild,heldout
14728.wav,Bernie Sanders,,genuine,bona-fide,release_in_the_wild/14728.wav,genuine,86,in_the_wild,heldout
471.wav,Milton Friedman,,genuine,bona-fide,release_in_the_wild/471.wav,genuine,87,in_the_wild,heldout
31611.wav,Frank Sinatra,,genuine,bona-fide,release_in_the_wild/31611.wav,genuine,88,in_the_wild,heldout
21341.wav,Robert Kardashian,,genuine,bona-fide,release_in_the_wild/21341.wav,genuine,89,in_the_wild,heldout
25713.wav,Barack Obama,,genuine,bona-fide,release_in_the_wild/25713.wav,genuine,90,in_the_wild,heldout
14739.wav,Alexandria Ocasio-Cortez,,genuine,bona-fide,release_in_the_wild/14739.wav,genuine,91,in_the_wild,heldout
10774.wav,Arnold Schwarzenegger,,genuine,bona-fide,release_in_the_wild/10774.wav,genuine,92,in_the_wild,heldout
16477.wav,Nick Offerman,,genuine,bona-fide,release_in_the_wild/16477.wav,genuine,93,in_the_wild,heldout
1284.wav,Norm MacDonald,,genuine,bona-fide,release_in_the_wild/1284.wav,genuine,94,in_the_wild,heldout
1693.wav,Bill Burr,,genuine,bona-fide,release_in_the_wild/1693.wav,genuine,95,in_the_wild,heldout
25903.wav,Kamala Harris,,genuine,bona-fide,release_in_the_wild/25903.wav,genuine,96,in_the_wild,heldout
19554.wav,George Carlin,,genuine,bona-fide,release_in_the_wild/19554.wav,genuine,97,in_the_wild,heldout
23842.wav,Jimmy Carter,,genuine,bona-fide,release_in_the_wild/23842.wav,genuine,98,in_the_wild,heldout
15859.wav,Louis Farrakhan,,genuine,bona-fide,release_in_the_wild/15859.wav,genuine,99,in_the_wild,heldout
25921.wav,Boris Johnson,,genuine,bona-fide,release_in_the_wild/25921.wav,genuine,100,in_the_wild,heldout
29043.wav,Winston Churchill,,genuine,bona-fide,release_in_the_wild/29043.wav,genuine,101,in_the_wild,heldout
10815.wav,Jerry Seinfeld,,genuine,bona-fide,release_in_the_wild/10815.wav,genuine,102,in_the_wild,heldout
6018.wav,William F. Buckley,,genuine,bona-fide,release_in_the_wild/6018.wav,genuine,103,in_the_wild,heldout
5255.wav,Dwight Eisenhower,,genuine,bona-fide,release_in_the_wild/5255.wav,genuine,104,in_the_wild,heldout
28108.wav,Bill Clinton,,genuine,bona-fide,release_in_the_wild/28108.wav,genuine,105,in_the_wild,heldout
17532.wav,Mr. Rogers,,genuine,bona-fide,release_in_the_wild/17532.wav,genuine,106,in_the_wild,heldout
12451.wav,Malcolm X,,genuine,bona-fide,release_in_the_wild/12451.wav,genuine,107,in_the_wild,heldout
51.wav,Dave Chappelle,,genuine,bona-fide,release_in_the_wild/51.wav,genuine,108,in_the_wild,heldout
28345.wav,Calvin Coolidge,,genuine,bona-fide,release_in_the_wild/28345.wav,genuine,109,in_the_wild,heldout
14577.wav,Donald Trump,,genuine,bona-fide,release_in_the_wild/14577.wav,genuine,110,in_the_wild,heldout
9629.wav,2Pac,,genuine,bona-fide,release_in_the_wild/9629.wav,genuine,111,in_the_wild,heldout
22472.wav,Ronald Reagan,,genuine,bona-fide,release_in_the_wild/22472.wav,genuine,112,in_the_wild,heldout
7730.wav,Mark Zuckerberg,,genuine,bona-fide,release_in_the_wild/7730.wav,genuine,113,in_the_wild,heldout
11874.wav,Ayn Rand,,genuine,bona-fide,release_in_the_wild/11874.wav,genuine,114,in_the_wild,heldout
4569.wav,Alec Guinness,,genuine,bona-fide,release_in_the_wild/4569.wav,genuine,115,in_the_wild,heldout
21772.wav,Jeff Goldblum,,genuine,bona-fide,release_in_the_wild/21772.wav,genuine,116,in_the_wild,heldout
4090.wav,Queen Elizabeth II,,genuine,bona-fide,release_in_the_wild/4090.wav,genuine,117,in_the_wild,heldout
6087.wav,Bob Ross,,genuine,bona-fide,release_in_the_wild/6087.wav,genuine,118,in_the_wild,heldout
25304.wav,Lyndon Johnson,,genuine,bona-fide,release_in_the_wild/25304.wav,genuine,119,in_the_wild,heldout
23922.wav,Christopher Hitchens,,genuine,bona-fide,release_in_the_wild/23922.wav,genuine,120,in_the_wild,heldout
5605.wav,Martin Luther King,,genuine,bona-fide,release_in_the_wild/5605.wav,genuine,121,in_the_wild,heldout
8921.wav,Richard Nixon,,genuine,bona-fide,release_in_the_wild/8921.wav,genuine,122,in_the_wild,heldout
11872.wav,Gilbert Gottfried,,genuine,bona-fide,release_in_the_wild/11872.wav,genuine,123,in_the_wild,heldout
8128.wav,Nelson Mandela,,genuine,bona-fide,release_in_the_wild/8128.wav,genuine,124,in_the_wild,heldout
28701.wav,John Cleese,,genuine,bona-fide,release_in_the_wild/28701.wav,genuine,125,in_the_wild,heldout
2261.wav,Adam Driver,,genuine,bona-fide,release_in_the_wild/2261.wav,genuine,126,in_the_wild,heldout
26731.wav,JFK,,genuine,bona-fide,release_in_the_wild/26731.wav,genuine,127,in_the_wild,heldout
13322.wav,FDR,,genuine,bona-fide,release_in_the_wild/13322.wav,genuine,128,in_the_wild,heldout
13621.wav,George W. Bush,,genuine,bona-fide,release_in_the_wild/13621.wav,genuine,129,in_the_wild,heldout
6615.wav,Scarlett Johansson,,genuine,bona-fide,release_in_the_wild/6615.wav,genuine,130,in_the_wild,heldout
22176.wav,The Notorious B.I.G.,,genuine,bona-fide,release_in_the_wild/22176.wav,genuine,131,in_the_wild,heldout
20158.wav,Louis C.K.,,genuine,bona-fide,release_in_the_wild/20158.wav,genuine,132,in_the_wild,heldout
5247.wav,Harry Truman,,genuine,bona-fide,release_in_the_wild/5247.wav,genuine,133,in_the_wild,heldout
15363.wav,Billie Eilish,,genuine,bona-fide,release_in_the_wild/15363.wav,genuine,134,in_the_wild,heldout
22067.wav,Alan Watts,,genuine,bona-fide,release_in_the_wild/22067.wav,genuine,135,in_the_wild,heldout
26503.wav,Kanye West,,genuine,bona-fide,release_in_the_wild/26503.wav,genuine,136,in_the_wild,heldout
11334.wav,Tucker Carlson,,genuine,bona-fide,release_in_the_wild/11334.wav,genuine,137,in_the_wild,heldout
2828.wav,Orson Welles,,genuine,bona-fide,release_in_the_wild/2828.wav,genuine,138,in_the_wild,heldout
22042.wav,Bernie Sanders,,genuine,bona-fide,release_in_the_wild/22042.wav,genuine,139,in_the_wild,heldout
5578.wav,Milton Friedman,,genuine,bona-fide,release_in_the_wild/5578.wav,genuine,140,in_the_wild,heldout
26225.wav,Frank Sinatra,,genuine,bona-fide,release_in_the_wild/26225.wav,genuine,141,in_the_wild,heldout
10313.wav,Robert Kardashian,,genuine,bona-fide,release_in_the_wild/10313.wav,genuine,142,in_the_wild,heldout
21838.wav,Barack Obama,,genuine,bona-fide,release_in_the_wild/21838.wav,genuine,143,in_the_wild,heldout
10841.wav,Alexandria Ocasio-Cortez,,genuine,bona-fide,release_in_the_wild/10841.wav,genuine,144,in_the_wild,heldout
11884.wav,Arnold Schwarzenegger,,genuine,bona-fide,release_in_the_wild/11884.wav,genuine,145,in_the_wild,heldout
21488.wav,Nick Offerman,,genuine,bona-fide,release_in_the_wild/21488.wav,genuine,146,in_the_wild,heldout
6252.wav,Norm MacDonald,,genuine,bona-fide,release_in_the_wild/6252.wav,genuine,147,in_the_wild,heldout
11794.wav,Bill Burr,,genuine,bona-fide,release_in_the_wild/11794.wav,genuine,148,in_the_wild,heldout
8125.wav,Kamala Harris,,genuine,bona-fide,release_in_the_wild/8125.wav,genuine,149,in_the_wild,heldout
797.wav,Scarlett Johansson,,synthetic,spoof,release_in_the_wild/797.wav,synthetic,150,in_the_wild,heldout
14440.wav,Tucker Carlson,,synthetic,spoof,release_in_the_wild/14440.wav,synthetic,151,in_the_wild,heldout
13241.wav,Queen Elizabeth II,,synthetic,spoof,release_in_the_wild/13241.wav,synthetic,152,in_the_wild,heldout
15958.wav,Mr. Rogers,,synthetic,spoof,release_in_the_wild/15958.wav,synthetic,153,in_the_wild,heldout
4410.wav,JFK,,synthetic,spoof,release_in_the_wild/4410.wav,synthetic,154,in_the_wild,heldout
5751.wav,Jeff Goldblum,,synthetic,spoof,release_in_the_wild/5751.wav,synthetic,155,in_the_wild,heldout
18603.wav,George Carlin,,synthetic,spoof,release_in_the_wild/18603.wav,synthetic,156,in_the_wild,heldout
28505.wav,Norm MacDonald,,synthetic,spoof,release_in_the_wild/28505.wav,synthetic,157,in_the_wild,heldout
9363.wav,Kamala Harris,,synthetic,spoof,release_in_the_wild/9363.wav,synthetic,158,in_the_wild,heldout
28676.wav,2Pac,,synthetic,spoof,release_in_the_wild/28676.wav,synthetic,159,in_the_wild,heldout
19431.wav,FDR,,synthetic,spoof,release_in_the_wild/19431.wav,synthetic,160,in_the_wild,heldout
586.wav,Robert Kardashian,,synthetic,spoof,release_in_the_wild/586.wav,synthetic,161,in_the_wild,heldout
30558.wav,Calvin Coolidge,,synthetic,spoof,release_in_the_wild/30558.wav,synthetic,162,in_the_wild,heldout
10786.wav,Adam Driver,,synthetic,spoof,release_in_the_wild/10786.wav,synthetic,163,in_the_wild,heldout
21355.wav,Billie Eilish,,synthetic,spoof,release_in_the_wild/21355.wav,synthetic,164,in_the_wild,heldout
18351.wav,Dave Chappelle,,synthetic,spoof,release_in_the_wild/18351.wav,synthetic,165,in_the_wild,heldout
13303.wav,Malcolm X,,synthetic,spoof,release_in_the_wild/13303.wav,synthetic,166,in_the_wild,heldout
10396.wav,Jimmy Carter,,synthetic,spoof,release_in_the_wild/10396.wav,synthetic,167,in_the_wild,heldout
28251.wav,The Notorious B.I.G.,,synthetic,spoof,release_in_the_wild/28251.wav,synthetic,168,in_the_wild,heldout
8970.wav,Lyndon Johnson,,synthetic,spoof,release_in_the_wild/8970.wav,synthetic,169,in_the_wild,heldout
25768.wav,Richard Nixon,,synthetic,spoof,release_in_the_wild/25768.wav,synthetic,170,in_the_wild,heldout
18343.wav,Nelson Mandela,,synthetic,spoof,release_in_the_wild/18343.wav,synthetic,171,in_the_wild,heldout
23264.wav,Jerry Seinfeld,,synthetic,spoof,release_in_the_wild/23264.wav,synthetic,172,in_the_wild,heldout
4594.wav,Alan Watts,,synthetic,spoof,release_in_the_wild/4594.wav,synthetic,173,in_the_wild,heldout
14249.wav,Winston Churchill,,synthetic,spoof,release_in_the_wild/14249.wav,synthetic,174,in_the_wild,heldout
23480.wav,Ayn Rand,,synthetic,spoof,release_in_the_wild/23480.wav,synthetic,175,in_the_wild,heldout
31047.wav,Martin Luther King,,synthetic,spoof,release_in_the_wild/31047.wav,synthetic,176,in_the_wild,heldout
15821.wav,Nick Offerman,,synthetic,spoof,release_in_the_wild/15821.wav,synthetic,177,in_the_wild,heldout
30908.wav,Ronald Reagan,,synthetic,spoof,release_in_the_wild/30908.wav,synthetic,178,in_the_wild,heldout
5518.wav,Bill Burr,,synthetic,spoof,release_in_the_wild/5518.wav,synthetic,179,in_the_wild,heldout
12567.wav,William F. Buckley,,synthetic,spoof,release_in_the_wild/12567.wav,synthetic,180,in_the_wild,heldout
10860.wav,Bernie Sanders,,synthetic,spoof,release_in_the_wild/10860.wav,synthetic,181,in_the_wild,heldout
15049.wav,Mark Zuckerberg,,synthetic,spoof,release_in_the_wild/15049.wav,synthetic,182,in_the_wild,heldout
11079.wav,Milton Friedman,,synthetic,spoof,release_in_the_wild/11079.wav,synthetic,183,in_the_wild,heldout
21172.wav,Boris Johnson,,synthetic,spoof,release_in_the_wild/21172.wav,synthetic,184,in_the_wild,heldout
11072.wav,Kanye West,,synthetic,spoof,release_in_the_wild/11072.wav,synthetic,185,in_the_wild,heldout
7755.wav,Dwight Eisenhower,,synthetic,spoof,release_in_the_wild/7755.wav,synthetic,186,in_the_wild,heldout
2280.wav,Mitch Hedberg,,synthetic,spoof,release_in_the_wild/2280.wav,synthetic,187,in_the_wild,heldout
15755.wav,Louis C.K.,,synthetic,spoof,release_in_the_wild/15755.wav,synthetic,188,in_the_wild,heldout
30515.wav,Louis Farrakhan,,synthetic,spoof,release_in_the_wild/30515.wav,synthetic,189,in_the_wild,heldout
27483.wav,Gilbert Gottfried,,synthetic,spoof,release_in_the_wild/27483.wav,synthetic,190,in_the_wild,heldout
20318.wav,Alexandria Ocasio-Cortez,,synthetic,spoof,release_in_the_wild/20318.wav,synthetic,191,in_the_wild,heldout
31276.wav,Donald Trump,,synthetic,spoof,release_in_the_wild/31276.wav,synthetic,192,in_the_wild,heldout
4184.wav,Bob Ross,,synthetic,spoof,release_in_the_wild/4184.wav,synthetic,193,in_the_wild,heldout
31331.wav,Christopher Hitchens,,synthetic,spoof,release_in_the_wild/31331.wav,synthetic,194,in_the_wild,heldout
443.wav,Arnold Schwarzenegger,,synthetic,spoof,release_in_the_wild/443.wav,synthetic,195,in_the_wild,heldout
14552.wav,Barack Obama,,synthetic,spoof,release_in_the_wild/14552.wav,synthetic,196,in_the_wild,heldout
25216.wav,George W. Bush,,synthetic,spoof,release_in_the_wild/25216.wav,synthetic,197,in_the_wild,heldout
782.wav,Harry Truman,,synthetic,spoof,release_in_the_wild/782.wav,synthetic,198,in_the_wild,heldout
19827.wav,Orson Welles,,synthetic,spoof,release_in_the_wild/19827.wav,synthetic,199,in_the_wild,heldout
2406.wav,Alec Guinness,,synthetic,spoof,release_in_the_wild/2406.wav,synthetic,200,in_the_wild,heldout
23118.wav,John Cleese,,synthetic,spoof,release_in_the_wild/23118.wav,synthetic,201,in_the_wild,heldout
26514.wav,Bill Clinton,,synthetic,spoof,release_in_the_wild/26514.wav,synthetic,202,in_the_wild,heldout
12100.wav,Frank Sinatra,,synthetic,spoof,release_in_the_wild/12100.wav,synthetic,203,in_the_wild,heldout
17178.wav,Scarlett Johansson,,synthetic,spoof,release_in_the_wild/17178.wav,synthetic,204,in_the_wild,heldout
9519.wav,Tucker Carlson,,synthetic,spoof,release_in_the_wild/9519.wav,synthetic,205,in_the_wild,heldout
11856.wav,Queen Elizabeth II,,synthetic,spoof,release_in_the_wild/11856.wav,synthetic,206,in_the_wild,heldout
7225.wav,Mr. Rogers,,synthetic,spoof,release_in_the_wild/7225.wav,synthetic,207,in_the_wild,heldout
6270.wav,JFK,,synthetic,spoof,release_in_the_wild/6270.wav,synthetic,208,in_the_wild,heldout
12590.wav,Jeff Goldblum,,synthetic,spoof,release_in_the_wild/12590.wav,synthetic,209,in_the_wild,heldout
24079.wav,George Carlin,,synthetic,spoof,release_in_the_wild/24079.wav,synthetic,210,in_the_wild,heldout
9726.wav,Norm MacDonald,,synthetic,spoof,release_in_the_wild/9726.wav,synthetic,211,in_the_wild,heldout
3381.wav,Kamala Harris,,synthetic,spoof,release_in_the_wild/3381.wav,synthetic,212,in_the_wild,heldout
24521.wav,2Pac,,synthetic,spoof,release_in_the_wild/24521.wav,synthetic,213,in_the_wild,heldout
19523.wav,FDR,,synthetic,spoof,release_in_the_wild/19523.wav,synthetic,214,in_the_wild,heldout
11276.wav,Robert Kardashian,,synthetic,spoof,release_in_the_wild/11276.wav,synthetic,215,in_the_wild,heldout
16562.wav,Calvin Coolidge,,synthetic,spoof,release_in_the_wild/16562.wav,synthetic,216,in_the_wild,heldout
6022.wav,Adam Driver,,synthetic,spoof,release_in_the_wild/6022.wav,synthetic,217,in_the_wild,heldout
30735.wav,Billie Eilish,,synthetic,spoof,release_in_the_wild/30735.wav,synthetic,218,in_the_wild,heldout
14992.wav,Dave Chappelle,,synthetic,spoof,release_in_the_wild/14992.wav,synthetic,219,in_the_wild,heldout
21942.wav,Malcolm X,,synthetic,spoof,release_in_the_wild/21942.wav,synthetic,220,in_the_wild,heldout
6929.wav,Jimmy Carter,,synthetic,spoof,release_in_the_wild/6929.wav,synthetic,221,in_the_wild,heldout
15268.wav,The Notorious B.I.G.,,synthetic,spoof,release_in_the_wild/15268.wav,synthetic,222,in_the_wild,heldout
25064.wav,Lyndon Johnson,,synthetic,spoof,release_in_the_wild/25064.wav,synthetic,223,in_the_wild,heldout
14253.wav,Richard Nixon,,synthetic,spoof,release_in_the_wild/14253.wav,synthetic,224,in_the_wild,heldout
26418.wav,Nelson Mandela,,synthetic,spoof,release_in_the_wild/26418.wav,synthetic,225,in_the_wild,heldout
25867.wav,Jerry Seinfeld,,synthetic,spoof,release_in_the_wild/25867.wav,synthetic,226,in_the_wild,heldout
18888.wav,Alan Watts,,synthetic,spoof,release_in_the_wild/18888.wav,synthetic,227,in_the_wild,heldout
7809.wav,Winston Churchill,,synthetic,spoof,release_in_the_wild/7809.wav,synthetic,228,in_the_wild,heldout
23279.wav,Ayn Rand,,synthetic,spoof,release_in_the_wild/23279.wav,synthetic,229,in_the_wild,heldout
9277.wav,Martin Luther King,,synthetic,spoof,release_in_the_wild/9277.wav,synthetic,230,in_the_wild,heldout
16104.wav,Nick Offerman,,synthetic,spoof,release_in_the_wild/16104.wav,synthetic,231,in_the_wild,heldout
8341.wav,Ronald Reagan,,synthetic,spoof,release_in_the_wild/8341.wav,synthetic,232,in_the_wild,heldout
10930.wav,Bill Burr,,synthetic,spoof,release_in_the_wild/10930.wav,synthetic,233,in_the_wild,heldout
17801.wav,William F. Buckley,,synthetic,spoof,release_in_the_wild/17801.wav,synthetic,234,in_the_wild,heldout
22682.wav,Bernie Sanders,,synthetic,spoof,release_in_the_wild/22682.wav,synthetic,235,in_the_wild,heldout
1795.wav,Mark Zuckerberg,,synthetic,spoof,release_in_the_wild/1795.wav,synthetic,236,in_the_wild,heldout
8478.wav,Milton Friedman,,synthetic,spoof,release_in_the_wild/8478.wav,synthetic,237,in_the_wild,heldout
17680.wav,Boris Johnson,,synthetic,spoof,release_in_the_wild/17680.wav,synthetic,238,in_the_wild,heldout
12832.wav,Kanye West,,synthetic,spoof,release_in_the_wild/12832.wav,synthetic,239,in_the_wild,heldout
31202.wav,Dwight Eisenhower,,synthetic,spoof,release_in_the_wild/31202.wav,synthetic,240,in_the_wild,heldout
1115.wav,Mitch Hedberg,,synthetic,spoof,release_in_the_wild/1115.wav,synthetic,241,in_the_wild,heldout
1122.wav,Louis C.K.,,synthetic,spoof,release_in_the_wild/1122.wav,synthetic,242,in_the_wild,heldout
16116.wav,Louis Farrakhan,,synthetic,spoof,release_in_the_wild/16116.wav,synthetic,243,in_the_wild,heldout
509.wav,Gilbert Gottfried,,synthetic,spoof,release_in_the_wild/509.wav,synthetic,244,in_the_wild,heldout
24909.wav,Alexandria Ocasio-Cortez,,synthetic,spoof,release_in_the_wild/24909.wav,synthetic,245,in_the_wild,heldout
14198.wav,Donald Trump,,synthetic,spoof,release_in_the_wild/14198.wav,synthetic,246,in_the_wild,heldout
11323.wav,Bob Ross,,synthetic,spoof,release_in_the_wild/11323.wav,synthetic,247,in_the_wild,heldout
24703.wav,Christopher Hitchens,,synthetic,spoof,release_in_the_wild/24703.wav,synthetic,248,in_the_wild,heldout
15165.wav,Arnold Schwarzenegger,,synthetic,spoof,release_in_the_wild/15165.wav,synthetic,249,in_the_wild,heldout
13492.wav,Barack Obama,,synthetic,spoof,release_in_the_wild/13492.wav,synthetic,250,in_the_wild,heldout
11389.wav,George W. Bush,,synthetic,spoof,release_in_the_wild/11389.wav,synthetic,251,in_the_wild,heldout
13601.wav,Harry Truman,,synthetic,spoof,release_in_the_wild/13601.wav,synthetic,252,in_the_wild,heldout
8896.wav,Orson Welles,,synthetic,spoof,release_in_the_wild/8896.wav,synthetic,253,in_the_wild,heldout
7506.wav,Alec Guinness,,synthetic,spoof,release_in_the_wild/7506.wav,synthetic,254,in_the_wild,heldout
30341.wav,John Cleese,,synthetic,spoof,release_in_the_wild/30341.wav,synthetic,255,in_the_wild,heldout
5719.wav,Bill Clinton,,synthetic,spoof,release_in_the_wild/5719.wav,synthetic,256,in_the_wild,heldout
27835.wav,Frank Sinatra,,synthetic,spoof,release_in_the_wild/27835.wav,synthetic,257,in_the_wild,heldout
19562.wav,Scarlett Johansson,,synthetic,spoof,release_in_the_wild/19562.wav,synthetic,258,in_the_wild,heldout
16247.wav,Tucker Carlson,,synthetic,spoof,release_in_the_wild/16247.wav,synthetic,259,in_the_wild,heldout
8947.wav,Queen Elizabeth II,,synthetic,spoof,release_in_the_wild/8947.wav,synthetic,260,in_the_wild,heldout
24893.wav,Mr. Rogers,,synthetic,spoof,release_in_the_wild/24893.wav,synthetic,261,in_the_wild,heldout
3336.wav,JFK,,synthetic,spoof,release_in_the_wild/3336.wav,synthetic,262,in_the_wild,heldout
24135.wav,George Carlin,,synthetic,spoof,release_in_the_wild/24135.wav,synthetic,263,in_the_wild,heldout
6477.wav,Norm MacDonald,,synthetic,spoof,release_in_the_wild/6477.wav,synthetic,264,in_the_wild,heldout
30362.wav,2Pac,,synthetic,spoof,release_in_the_wild/30362.wav,synthetic,265,in_the_wild,heldout
403.wav,FDR,,synthetic,spoof,release_in_the_wild/403.wav,synthetic,266,in_the_wild,heldout
28481.wav,Robert Kardashian,,synthetic,spoof,release_in_the_wild/28481.wav,synthetic,267,in_the_wild,heldout
26923.wav,Calvin Coolidge,,synthetic,spoof,release_in_the_wild/26923.wav,synthetic,268,in_the_wild,heldout
20975.wav,Adam Driver,,synthetic,spoof,release_in_the_wild/20975.wav,synthetic,269,in_the_wild,heldout
24105.wav,Billie Eilish,,synthetic,spoof,release_in_the_wild/24105.wav,synthetic,270,in_the_wild,heldout
16187.wav,Dave Chappelle,,synthetic,spoof,release_in_the_wild/16187.wav,synthetic,271,in_the_wild,heldout
8136.wav,Malcolm X,,synthetic,spoof,release_in_the_wild/8136.wav,synthetic,272,in_the_wild,heldout
27597.wav,Jimmy Carter,,synthetic,spoof,release_in_the_wild/27597.wav,synthetic,273,in_the_wild,heldout
16417.wav,The Notorious B.I.G.,,synthetic,spoof,release_in_the_wild/16417.wav,synthetic,274,in_the_wild,heldout
17232.wav,Lyndon Johnson,,synthetic,spoof,release_in_the_wild/17232.wav,synthetic,275,in_the_wild,heldout
2161.wav,Richard Nixon,,synthetic,spoof,release_in_the_wild/2161.wav,synthetic,276,in_the_wild,heldout
2577.wav,Nelson Mandela,,synthetic,spoof,release_in_the_wild/2577.wav,synthetic,277,in_the_wild,heldout
24077.wav,Jerry Seinfeld,,synthetic,spoof,release_in_the_wild/24077.wav,synthetic,278,in_the_wild,heldout
15544.wav,Alan Watts,,synthetic,spoof,release_in_the_wild/15544.wav,synthetic,279,in_the_wild,heldout
11504.wav,Winston Churchill,,synthetic,spoof,release_in_the_wild/11504.wav,synthetic,280,in_the_wild,heldout
20311.wav,Ayn Rand,,synthetic,spoof,release_in_the_wild/20311.wav,synthetic,281,in_the_wild,heldout
9058.wav,Martin Luther King,,synthetic,spoof,release_in_the_wild/9058.wav,synthetic,282,in_the_wild,heldout
10925.wav,Nick Offerman,,synthetic,spoof,release_in_the_wild/10925.wav,synthetic,283,in_the_wild,heldout
7447.wav,Ronald Reagan,,synthetic,spoof,release_in_the_wild/7447.wav,synthetic,284,in_the_wild,heldout
20452.wav,Bill Burr,,synthetic,spoof,release_in_the_wild/20452.wav,synthetic,285,in_the_wild,heldout
19908.wav,William F. Buckley,,synthetic,spoof,release_in_the_wild/19908.wav,synthetic,286,in_the_wild,heldout
25352.wav,Bernie Sanders,,synthetic,spoof,release_in_the_wild/25352.wav,synthetic,287,in_the_wild,heldout
427.wav,Mark Zuckerberg,,synthetic,spoof,release_in_the_wild/427.wav,synthetic,288,in_the_wild,heldout
31704.wav,Milton Friedman,,synthetic,spoof,release_in_the_wild/31704.wav,synthetic,289,in_the_wild,heldout
27841.wav,Boris Johnson,,synthetic,spoof,release_in_the_wild/27841.wav,synthetic,290,in_the_wild,heldout
14034.wav,Kanye West,,synthetic,spoof,release_in_the_wild/14034.wav,synthetic,291,in_the_wild,heldout
17257.wav,Dwight Eisenhower,,synthetic,spoof,release_in_the_wild/17257.wav,synthetic,292,in_the_wild,heldout
30470.wav,Mitch Hedberg,,synthetic,spoof,release_in_the_wild/30470.wav,synthetic,293,in_the_wild,heldout
26003.wav,Louis C.K.,,synthetic,spoof,release_in_the_wild/26003.wav,synthetic,294,in_the_wild,heldout
23053.wav,Louis Farrakhan,,synthetic,spoof,release_in_the_wild/23053.wav,synthetic,295,in_the_wild,heldout
20373.wav,Gilbert Gottfried,,synthetic,spoof,release_in_the_wild/20373.wav,synthetic,296,in_the_wild,heldout
14227.wav,Alexandria Ocasio-Cortez,,synthetic,spoof,release_in_the_wild/14227.wav,synthetic,297,in_the_wild,heldout
12804.wav,Donald Trump,,synthetic,spoof,release_in_the_wild/12804.wav,synthetic,298,in_the_wild,heldout
9154.wav,Bob Ross,,synthetic,spoof,release_in_the_wild/9154.wav,synthetic,299,in_the_wild,heldout

In [ ]:
%%writefile inspected_clip_ids.txt
10023.wav
10055.wav
10241.wav
10313.wav
10396.wav
10400.wav
10476.wav
10496.wav
10516.wav
10756.wav
10774.wav
10786.wav
10815.wav
10841.wav
10860.wav
10864.wav
10902.wav
10925.wav
10930.wav
11044.wav
11072.wav
11079.wav
11124.wav
1115.wav
1122.wav
11225.wav
11276.wav
11323.wav
11334.wav
11389.wav
11504.wav
1159.wav
11729.wav
11794.wav
11835.wav
11856.wav
11872.wav
11874.wav
11884.wav
1194.wav
12100.wav
12142.wav
12185.wav
12368.wav
12451.wav
12567.wav
12590.wav
12681.wav
12720.wav
12789.wav
12804.wav
12832.wav
1284.wav
13027.wav
1309.wav
13152.wav
1323.wav
13241.wav
13303.wav
13322.wav
13421.wav
13492.wav
13583.wav
13586.wav
13601.wav
13621.wav
13659.wav
13957.wav
14034.wav
14198.wav
14227.wav
14249.wav
14253.wav
1428.wav
14378.wav
14411.wav
14440.wav
14552.wav
14577.wav
14579.wav
14691.wav
14728.wav
14739.wav
14782.wav
14840.wav
14992.wav
15049.wav
15148.wav
15165.wav
15195.wav
15268.wav
15363.wav
15381.wav
15408.wav
1542.wav
15467.wav
15487.wav
15544.wav
1558.wav
15660.wav
1575.wav
15755.wav
15821.wav
15829.wav
15859.wav
15860.wav
15865.wav
15958.wav
16034.wav
16049.wav
16104.wav
16116.wav
16164.wav
16187.wav
16247.wav
16256.wav
16417.wav
16428.wav
1645.wav
16477.wav
16528.wav
16562.wav
16681.wav
16744.wav
16803.wav
16827.wav
16855.wav
1693.wav
17174.wav
17178.wav
17232.wav
17257.wav
17368.wav
1747.wav
17532.wav
17680.wav
17801.wav
17861.wav
17921.wav
1795.wav
18011.wav
18343.wav
18351.wav
1845.wav
18603.wav
18831.wav
18833.wav
18847.wav
18888.wav
19027.wav
19041.wav
19088.wav
19242.wav
19265.wav
19431.wav
19523.wav
19554.wav
19562.wav
1959.wav
19715.wav
19803.wav
19827.wav
19890.wav
19894.wav
19908.wav
20156.wav
20158.wav
20311.wav
20318.wav
20336.wav
20373.wav
20452.wav
20495.wav
20520.wav
20703.wav
20736.wav
20859.wav
20952.wav
20975.wav
21050.wav
21100.wav
21123.wav
21172.wav
21238.wav
21255.wav
21273.wav
21298.wav
21309.wav
21329.wav
21341.wav
21355.wav
21372.wav
21419.wav
21464.wav
21486.wav
21488.wav
2161.wav
2165.wav
21657.wav
21772.wav
21838.wav
21906.wav
21942.wav
2197.wav
22036.wav
22042.wav
22067.wav
22081.wav
22176.wav
22217.wav
22274.wav
22330.wav
22423.wav
22472.wav
2261.wav
22682.wav
22745.wav
2280.wav
22936.wav
23053.wav
23118.wav
23195.wav
23264.wav
23279.wav
23318.wav
234.wav
23480.wav
23566.wav
23722.wav
23842.wav
23922.wav
2406.wav
24077.wav
24079.wav
24105.wav
24135.wav
24162.wav
24262.wav
24427.wav
24521.wav
24562.wav
24648.wav
24703.wav
24719.wav
24780.wav
24893.wav
24909.wav
24982.wav
25061.wav
25064.wav
25073.wav
25114.wav
2513.wav
25138.wav
25160.wav
25216.wav
25229.wav
25304.wav
25352.wav
25438.wav
25610.wav
25713.wav
25765.wav
25768.wav
2577.wav
25771.wav
25781.wav
25867.wav
25903.wav
25912.wav
25921.wav
25954.wav
25957.wav
26003.wav
26179.wav
26225.wav
26297.wav
26404.wav
26418.wav
26460.wav
26503.wav
26514.wav
26603.wav
26640.wav
26719.wav
26731.wav
26765.wav
26923.wav
26924.wav
26962.wav
27119.wav
27151.wav
27252.wav
2726.wav
27364.wav
27483.wav
27526.wav
27536.wav
27564.wav
27597.wav
27706.wav
27835.wav
27841.wav
27937.wav
27941.wav
27970.wav
28015.wav
2803.wav
28108.wav
28164.wav
2817.wav
28222.wav
28251.wav
28252.wav
2828.wav
28345.wav
28481.wav
28491.wav
28505.wav
28508.wav
28676.wav
28701.wav
28742.wav
28755.wav
28925.wav
29043.wav
29059.wav
2906.wav
29081.wav
29120.wav
29129.wav
2923.wav
29328.wav
29606.wav
29657.wav
29767.wav
29785.wav
29835.wav
29854.wav
29865.wav
29927.wav
29975.wav
30066.wav
30117.wav
30246.wav
30341.wav
3035.wav
30362.wav
30389.wav
30441.wav
30470.wav
30515.wav
30558.wav
30559.wav
30610.wav
30665.wav
30695.wav
30700.wav
30735.wav
30908.wav
31047.wav
31078.wav
31118.wav
31176.wav
31202.wav
31222.wav
31276.wav
31284.wav
31331.wav
31611.wav
31704.wav
31752.wav
3236.wav
3336.wav
3381.wav
344.wav
3520.wav
3545.wav
3708.wav
3717.wav
3910.wav
3936.wav
403.wav
4084.wav
4090.wav
4129.wav
4184.wav
4199.wav
4266.wav
427.wav
4292.wav
4410.wav
443.wav
4541.wav
4569.wav
4590.wav
4594.wav
4605.wav
4615.wav
4707.wav
471.wav
4852.wav
4914.wav
5020.wav
5044.wav
5052.wav
509.wav
51.wav
5247.wav
5255.wav
5268.wav
5309.wav
5319.wav
5387.wav
5452.wav
5483.wav
5490.wav
5511.wav
5518.wav
5578.wav
5605.wav
5705.wav
5719.wav
5751.wav
5758.wav
5784.wav
5792.wav
586.wav
5893.wav
5937.wav
5967.wav
6018.wav
6022.wav
6075.wav
6087.wav
612.wav
6147.wav
6154.wav
6252.wav
6270.wav
6291.wav
6477.wav
6499.wav
6606.wav
6615.wav
6649.wav
6718.wav
6817.wav
6929.wav
6971.wav
703.wav
7198.wav
7225.wav
7293.wav
7447.wav
7506.wav
7730.wav
7755.wav
777.wav
7809.wav
7818.wav
782.wav
7852.wav
7894.wav
7948.wav
797.wav
8119.wav
8125.wav
8128.wav
8136.wav
8341.wav
8478.wav
8504.wav
8604.wav
8618.wav
8837.wav
8846.wav
8896.wav
8921.wav
8947.wav
8970.wav
9027.wav
9058.wav
9107.wav
9154.wav
9157.wav
9240.wav
9266.wav
9277.wav
9278.wav
9363.wav
9375.wav
9502.wav
9519.wav
9532.wav
9548.wav
9629.wav
9726.wav
9851.wav
9924.wav

In [ ]:
# 3. Imports, provenance, environment, front-end availability
import os, json, time, hashlib, platform, datetime, importlib, traceback, shutil, csv
import numpy as np, pandas as pd, torch
sys.path.insert(0, os.getcwd())
import dhwani_infer as dh, eval_lib as el, antideepfake_hf as ad, degrade as dg, frontend as fe
for m in (dh, el, ad, dg, fe): importlib.reload(m)
EXPECTED = {
    "dhwani_infer.py": "8a7c9c8f4275fb27a0d6ba3dde43aaeedf98974b7057378c37122467ba756077",
    "eval_lib.py": "c96b5c999c20f6bb88f43d6cd4aee9afd8b3c63f1d4c5b89d87b5869bef19af2",
    "antideepfake_hf.py": "fba4d117443e6ff10dd0b7fdd64f89cff94960a73b8f0a85799ec884eb2b3db5",
    "degrade.py": "26b94163c1d449972db72b2bd547ec972e66180cfd6677fe1e3a242b344571f2",
    "frontend.py": "1b50820e59a6f5dd6bd3a6197a0229fad5da2e2b81f2f152d66b73d5b4658e1f"
}
CODE_SHA256 = {f: hashlib.sha256((open(f, encoding="utf-8").read().rstrip() + "\n").encode()).hexdigest() for f in EXPECTED}
assert CODE_SHA256 == EXPECTED, CODE_SHA256
assert FROZEN_THRESHOLD == 0.5
INSPECTED = set(l.strip() for l in open("inspected_clip_ids.txt") if l.strip()); assert len(INSPECTED) == 500
DEV_MAN = list(csv.DictReader(open("manifest_dev_heldout_in_the_wild.csv"))); assert len(DEV_MAN) == 300
AVAIL = fe.available(); print("front-ends:", AVAIL)
FRONTENDS_RUN = [f for f in FRONTENDS if AVAIL.get(f) is True]
print("running front-ends:", FRONTENDS_RUN, "| unavailable:", {f: AVAIL.get(f) for f in FRONTENDS if AVAIL.get(f) is not True})
FFMPEG = dg.ffmpeg_available(); print("ffmpeg:", FFMPEG)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
import transformers, librosa, soundfile, scipy, noisereduce
ENV = dict(python=platform.python_version(), torch=torch.__version__, transformers=transformers.__version__, numpy=np.__version__,
           librosa=librosa.__version__, soundfile=soundfile.__version__, scipy=scipy.__version__, noisereduce=noisereduce.__version__,
           device=DEVICE, gpu=torch.cuda.get_device_name(0) if DEVICE == "cuda" else None, ffmpeg=FFMPEG, frontends_available=AVAIL,
           dfn_install=DFN_INSTALL, code_sha256=CODE_SHA256)
print(json.dumps(ENV, indent=1, default=str))

In [ ]:
# 4. Candidate (frozen weights; reuse runtime object if present)
from huggingface_hub import hf_hub_download
if "cand" in globals() and getattr(cand, "report", None):
    print("reusing loaded candidate")
else:
    if USE_DRIVE:
        from google.colab import drive; drive.mount("/content/drive")
    try: AD_PATH = hf_hub_download(ad.MODEL_REPO, ad.MODEL_FILE, revision=ad.MODEL_REVISION, local_files_only=True)
    except Exception: AD_PATH = hf_hub_download(ad.MODEL_REPO, ad.MODEL_FILE, revision=ad.MODEL_REVISION)
    assert os.path.getsize(AD_PATH) == ad.MODEL_BYTES and dh.sha256_file(AD_PATH) == ad.MODEL_SHA256
    AD_CFG = hf_hub_download(ad.MODEL_REPO, "config.json", revision=ad.MODEL_REVISION)
    cand = ad.AntiDeepfakeHF.load(AD_PATH, os.path.dirname(AD_CFG), device=DEVICE)
print({k: cand.report[k] for k in ("n_encoder_mapped", "n_params_encoder")})
BASE = os.path.join(DRIVE_DIR if USE_DRIVE else WORK, "runs")
utc = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ")
RUN_NAME = f"{utc}_denoiser_frontend_itw"; RUN_DIR = os.path.join(BASE, RUN_NAME); os.makedirs(RUN_DIR, exist_ok=True)
el.jdump({"settings": dict(FRONTENDS=FRONTENDS, FRONTENDS_RUN=FRONTENDS_RUN, OFFICIAL_MARGIN_PP=OFFICIAL_MARGIN_PP, DEV_SELECTION_MARGIN_PP=DEV_SELECTION_MARGIN_PP,
          margin_note="official 2 pp is a provisional interpretation of 'within 2 points'; 1 pp is a development selection margin only", FROZEN_THRESHOLD=FROZEN_THRESHOLD,
          REQUIRED=REQUIRED, FRESH_SEED=FRESH_SEED, FRESH_N=FRESH_N, N_BOOT=N_BOOT, frontend_specs=fe.FRONTENDS, conditions=dg.CONDITIONS,
          noise_seed=dg.NOISE_SEED, candidate=dict(repo=ad.MODEL_REPO, revision=ad.MODEL_REVISION, sha256=ad.MODEL_SHA256)),
          "env": ENV, "started_utc": utc}, os.path.join(RUN_DIR, "run_config.json"))
print(RUN_DIR)

In [ ]:
# 5. Shared scoring loop: decode once -> degrade -> front-end -> validate -> score (paired; every failure recorded)
AUD = os.path.join(WORK, "audio"); os.makedirs(os.path.join(AUD, "heldout"), exist_ok=True); os.makedirs(os.path.join(AUD, "fresh"), exist_ok=True)
_zf = None
def ensure_audio(m, sub):
    global _zf
    p = os.path.join(AUD, sub, os.path.basename(m["archive_member"]))
    if not os.path.exists(p):
        if _zf is None:
            src = el.SOURCES["in_the_wild"]; _zf, _ = el.open_remote_zip(src["url"], src["bytes"])
        with open(p, "wb") as f: f.write(_zf.read(m["archive_member"]))
    return p

def run_stage(manifest, sub, frontends, path, stage):
    done = set()
    if os.path.exists(path):
        done = {(r["clip_id"], r["condition"], r["frontend"]) for r in map(json.loads, open(path)) if r}
    t0 = time.time()
    with open(path, "a", encoding="utf-8") as out:
        for i, m in enumerate(manifest):
            base = {"stage": stage, "clip_id": m["clip_id"], "label": m["label"], "speaker_id": m["speaker_id"]}
            try:
                y, sr = dh.load_audio(ensure_audio(m, sub)); x16 = dh.to_mono_16k(y, sr); x16, info = dh.validate(x16); ok = True
            except Exception as e:
                ok, info = False, {"reject_reason": f"{type(e).__name__}: {e}"}
            for cond in REQUIRED:
                xd = None
                for f in frontends:
                    if (m["clip_id"], cond, f) in done: continue
                    row = dict(base, condition=cond, frontend=f, clean_eligible=ok, duration_s=info.get("duration_s"))
                    if not ok:
                        row.update(status="rejected_clean", reject_reason=info["reject_reason"]); out.write(json.dumps(row) + "\n"); continue
                    try:
                        if xd is None: xd = dg.apply(cond, x16, clip_id=m["clip_id"])
                        t1 = time.perf_counter(); xf = fe.apply(f, xd); row["frontend_ms"] = round((time.perf_counter() - t1) * 1000, 1)
                        xv, vinfo = dh.validate(xf)
                        c = ad.score_clip(cand, xv)
                        row.update(status="ok", cand_fake_score=c["fake_score_whole"], cand_logits=json.dumps(c["logits_fake_real"]),
                                   out_peak_dbfs=vinfo["peak_dbfs"], out_rms_dbfs=vinfo["rms_dbfs"])
                    except dh.AudioRejected as e:
                        row.update(status="rejected_after_frontend", reject_reason=str(e), reject_category=el.reject_category(str(e)))
                    except Exception as e:
                        row.update(status="failed", reject_reason=f"{type(e).__name__}: {str(e)[:300]}")
                    out.write(json.dumps(row, default=str) + "\n"); out.flush()
            if (i + 1) % 25 == 0 or i + 1 == len(manifest):
                e = time.time() - t0; print(f"[{stage}] {i+1}/{len(manifest)} {e/60:.1f} min, ETA {e/(i+1)*(len(manifest)-i-1)/60:.1f} min")
    return pd.DataFrame([json.loads(l) for l in open(path) if l.strip()]).drop_duplicates(["clip_id", "condition", "frontend"], keep="last")

def eer_table(P, frontends):
    OK = P[P.status == "ok"]
    paired = set.intersection(*[set(OK[(OK.condition == c) & (OK.frontend == f)].clip_id) for c in REQUIRED for f in frontends])
    res, eers = {}, {}
    for f in frontends:
        res[f], eers[f] = {}, {}
        for c in REQUIRED:
            d = OK[(OK.condition == c) & (OK.frontend == f) & OK.clip_id.isin(paired)]
            y = (d.label == "synthetic").astype(int).tolist(); s = d.cand_fake_score.astype(float).tolist()
            m = el.evaluate(y, s, FROZEN_THRESHOLD, n_boot=N_BOOT, seed=FRESH_SEED) if len(set(y)) == 2 else None
            res[f][c] = m; eers[f][c] = m["threshold_free"]["eer"] if m else None
    return paired, res, eers

def print_table(res, eers, frontends, title):
    print(f"\n===== {title} =====")
    for f in frontends:
        print(f"-- front-end: {f}")
        for c in REQUIRED:
            m = res[f][c]
            if m is None: print(f"   {c:15s} n/a"); continue
            tf, bs, a = m["threshold_free"], m["bootstrap95"], m["at_threshold"]
            print(f"   {c:15s} n={m['n_scored']:3d} EER {tf['eer']*100:5.2f}% [{bs['eer'][0]*100:4.1f},{bs['eer'][1]*100:4.1f}] AUC {tf['roc_auc']:.4f} "
                  f"| @0.5 acc {a['accuracy']:.3f} rec {a['synthetic_recall']:.3f} FPR {a['genuine_fpr']:.3f}")
        d = {c: eers[f][c] for c in REQUIRED if c != "clean"}
        if all(v is not None for v in d.values()) and eers[f]["clean"]:
            w = max(d, key=d.get); print(f"   worst {w} / clean = {d[w]/eers[f]['clean']!r} (unrounded)")

In [ ]:
# 6. STAGE A — development evidence: all front-ends x required conditions on the 244 inspected clips
DEV_PATH = os.path.join(RUN_DIR, "stageA_dev_predictions.jsonl")
PA = run_stage(DEV_MAN, "heldout", FRONTENDS_RUN, DEV_PATH, "A_dev")
covA = PA.groupby(["frontend", "condition"]).status.value_counts().unstack(fill_value=0)
print("Stage A coverage:\n", covA.to_string())
pairedA, resA, eersA = eer_table(PA, FRONTENDS_RUN)
print(f"paired dev clips: {len(pairedA)}")
print_table(resA, eersA, FRONTENDS_RUN, "STAGE A (development evidence; no decisions on these numbers except the pre-registered freeze rule)")

In [ ]:
# 7. STAGE A — artefact-removal check on CLEAN audio + pre-registered freeze
OKA = PA[(PA.status == "ok") & (PA.condition == "clean") & PA.clip_id.isin(pairedA)]
base = OKA[OKA.frontend == "none"].set_index("clip_id").cand_fake_score.astype(float)
ARTEFACT = {}
for f in FRONTENDS_RUN:
    if f == "none": continue
    d = OKA[OKA.frontend == f].set_index("clip_id").cand_fake_score.astype(float)
    lab = OKA[OKA.frontend == f].set_index("clip_id").label
    j = base.index.intersection(d.index); delta = d[j] - base[j]
    syn, gen = delta[lab[j] == "synthetic"], delta[lab[j] == "genuine"]
    ARTEFACT[f] = {"n_pairs": int(len(j)),
                   "clean_eer_none": eersA["none"]["clean"], "clean_eer_frontend": eersA[f]["clean"],
                   "clean_eer_delta_pp": (eersA[f]["clean"] - eersA["none"]["clean"]) * 100,
                   "synthetic_median_delta": float(syn.median()), "synthetic_frac_dropped_over_0.25": float((syn < -0.25).mean()),
                   "synthetic_flips_to_genuine_at_0.5": int(((d[j] < 0.5) & (base[j] >= 0.5) & (lab[j] == "synthetic")).sum()),
                   "genuine_median_delta": float(gen.median()), "genuine_flips_to_synthetic_at_0.5": int(((d[j] >= 0.5) & (base[j] < 0.5) & (lab[j] == "genuine")).sum())}
print("Artefact-removal check on clean audio (front-end vs none, same clips):")
print(pd.DataFrame(ARTEFACT).T.to_string())
FREEZE = el.frontend_selection(eersA, baseline="none", margin_pp=DEV_SELECTION_MARGIN_PP, required=tuple(REQUIRED), order=[f for f in FRONTENDS_RUN if f != "none"])
print("\nPre-registered freeze rule (development selection margin", DEV_SELECTION_MARGIN_PP, "pp; stricter than the provisional official 2 pp):", FREEZE["rule"])
print(pd.DataFrame(FREEZE["table"]).T.to_string())
FROZEN = FREEZE["frozen_frontend"]
print("\nFROZEN FRONT-END:", FROZEN if FROZEN else "NONE (no front-end is eligible AND better than baseline -> denoiser experiment FAILED at stage A; proceed to fine-tuning plan)")
el.jdump({"artefact_check": ARTEFACT, "freeze": FREEZE, "frozen_frontend": FROZEN, "paired_dev_n": len(pairedA), "dev_eers": eersA,
          "dev_metrics": resA}, os.path.join(RUN_DIR, "stageA_summary.json"))

In [ ]:
# 8. STAGE B — fresh selection (excludes all 500 inspected IDs), then baseline + frozen front-end on required conditions
src = el.SOURCES["in_the_wild"]
if _zf is None: _zf, _ = el.open_remote_zip(src["url"], src["bytes"])
names = _zf.namelist(); meta_m = el.find_member(names, src["meta_suffix"]); prefix = meta_m[:-len("meta.csv")]
recs = el.parse_itw_meta(_zf.read(meta_m).decode("utf-8")); nameset = set(names)
for r in recs: r["archive_member"] = prefix + r["clip_id"] if prefix + r["clip_id"] in nameset else ""
FRESH, fmeta = el.select_subset(recs, "in_the_wild", FRESH_SEED, FRESH_N, n_synthetic=FRESH_N, exclude_clip_ids=INSPECTED)
assert not ({r["clip_id"] for r in FRESH} & INSPECTED)
fmeta["limitations"] = ["speakers shared with inspected sets (all 54; speaker-disjoint impossible on In-the-Wild)",
                        "candidate trained on VoxCeleb2 celebrity speech: speaker/recording overlap with In-the-Wild possible, unverified",
                        "In-the-Wild not in the candidate's training table (card)"]
pd.DataFrame(FRESH).to_csv(os.path.join(RUN_DIR, "manifest_fresh_in_the_wild.csv"), index=False)
el.jdump(fmeta, os.path.join(RUN_DIR, "selection_fresh_in_the_wild.json"))
print("fresh:", fmeta["n_selected"], fmeta["selected"]["by_label"], "excluded", fmeta["exclusion"], "| clip-ID sha256", fmeta["clip_id_list_sha256"][:16])
FRONTENDS_B = ["none"] + ([FROZEN] if FROZEN else [])
FRESH_PATH = os.path.join(RUN_DIR, "stageB_fresh_predictions.jsonl")
PB = run_stage(FRESH, "fresh", FRONTENDS_B, FRESH_PATH, "B_fresh")
covB = PB.groupby(["frontend", "condition"]).status.value_counts().unstack(fill_value=0)
covB["selected"] = covB.sum(axis=1); covB["coverage"] = (covB.get("ok", 0) / covB["selected"]).round(4)
print("Stage B coverage (every rejection shown):\n", covB.to_string())
pairedB, resB, eersB = eer_table(PB, FRONTENDS_B)
print(f"paired fresh clips: {len(pairedB)}")
print_table(resB, eersB, FRONTENDS_B, "STAGE B (fresh clips; threshold 0.5 frozen; EER thresholds descriptive only)")

In [ ]:
# 9. STAGE B — verdicts: official ratio (unrounded) per pipeline + clean-margin check
VERDICTS = {}
for f in FRONTENDS_B:
    v = el.robustness_verdict({c: eersB[f][c] for c in REQUIRED}, required=tuple(REQUIRED), n_pairs=len(pairedB))
    v["ratio_unrounded"] = v.get("ratio"); VERDICTS[f] = v
    print(f"\n[{f}] verdict {v['verdict']} | worst {v.get('worst_condition')} EER {v.get('worst_eer')} | clean EER {eersB[f]['clean']} | "
          f"ratio (unrounded) {v.get('ratio')!r} | EER(8k)-EER(clean) {v.get('eer_8k_minus_clean_pp')} pp")
if FROZEN:
    MARGIN = el.clean_margin_check(eersB[FROZEN]["clean"], eersB["none"]["clean"], OFFICIAL_MARGIN_PP)
    MARGIN["basis"] = "provisional interpretation of the supplied requirement 'Phase-1 EERs must remain within 2 points' as 2 percentage points; pending organiser clarification"
    MARGIN_DEV = el.clean_margin_check(eersB[FROZEN]["clean"], eersB["none"]["clean"], DEV_SELECTION_MARGIN_PP)
    print("\nClean-margin check (fresh, official provisional 2 pp):", MARGIN)
    print("Clean-margin check (fresh, stricter development 1 pp, informational):", MARGIN_DEV)
    final = ("PASS" if VERDICTS[FROZEN]["verdict"] == "PASS" and MARGIN["within_margin"] else
             ("FAIL (ratio ok but clean EER worsened beyond margin: not accepted)" if VERDICTS[FROZEN]["verdict"] == "PASS" else VERDICTS[FROZEN]["verdict"]))
    print(f"\nFINAL (frozen front-end '{FROZEN}'): {final}")
    abs_base = {c: round(eersB["none"][c] * 100, 2) for c in REQUIRED}; abs_fe = {c: round(eersB[FROZEN][c] * 100, 2) for c in REQUIRED}
    print("   absolute EERs (%) fresh: baseline", abs_base, " with front-end", abs_fe)
else:
    MARGIN, MARGIN_DEV, final = None, None, "NOT APPLICABLE: no front-end frozen at stage A; fresh baseline numbers reported only"
    print("\nFINAL:", final)

In [ ]:
# 10. Save (no audio) + download
PA.to_csv(os.path.join(RUN_DIR, "stageA_dev_predictions.csv"), index=False); PB.to_csv(os.path.join(RUN_DIR, "stageB_fresh_predictions.csv"), index=False)
covA.to_csv(os.path.join(RUN_DIR, "stageA_coverage.csv")); covB.to_csv(os.path.join(RUN_DIR, "stageB_coverage.csv"))
el.jdump({"run_name": RUN_NAME, "frozen_frontend": FROZEN, "official_margin_pp_provisional": OFFICIAL_MARGIN_PP, "dev_selection_margin_pp": DEV_SELECTION_MARGIN_PP,
          "frozen_threshold": FROZEN_THRESHOLD,
          "stageA": {"paired_n": len(pairedA), "eers": eersA, "artefact_check": ARTEFACT, "freeze": FREEZE},
          "stageB": {"paired_n": len(pairedB), "eers": eersB, "metrics": resB, "verdicts": VERDICTS, "clean_margin_official": MARGIN, "clean_margin_dev": MARGIN_DEV, "final": final,
                     "selection": fmeta, "coverage": covB.reset_index().to_dict("records")},
          "env": ENV, "finished_utc": datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ")}, os.path.join(RUN_DIR, "metrics.json"))
zip_path = shutil.make_archive(os.path.join(WORK, RUN_NAME), "zip", RUN_DIR); print(sorted(os.listdir(RUN_DIR)), zip_path)
try:
    from google.colab import files; files.download(zip_path)
except ImportError: pass

## Report back
Zip + executed notebook → `evaluation/runs/<run_name>/`. Paste cells 3 (front-end availability + ffmpeg), 6, 7, 8, 9.

## If the final verdict is not PASS
The denoiser experiment is recorded as failed and the next step is **notebook 07: noise-augmented fine-tuning** (not built or started yet):
start from the verified candidate weights; noise + codec augmentation on **both** classes; separate training and validation data with grouped splits
(speaker / generator where metadata permits); checkpoint selection by **validation EER**; cross-entropy retained; deployment threshold chosen on validation only;
final evaluation on a new fresh selection excluding every inspected ID (now 800 after this run) with the same five conditions and the same clean-margin rule.